In [2]:
# Setup: GPU + a Qwen3-capable transformers (needs >=4.51)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!pip install -q -U "transformers>=4.51.0" accelerate
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available())

NVIDIA A100-SXM4-40GB, 40960 MiB, 6 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 124.9 MB/s eta 0:00:00
torch 2.11.0+cu128 | transformers 5.14.1 | cuda True


In [3]:
# Environment — must run BEFORE anything imports huggingface_hub (phase 6 RECIPE stage 0).
# 1. HF_HUB_DISABLE_XET: without it the safetensors shards hang at 0 bytes.
# 2. HF_TOKEN from the Colab secrets vault, read early. Never print the token itself.
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok
        os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
    print("HF_TOKEN present:", bool(tok))
except Exception as e:
    print("no colab secrets:", type(e).__name__)

HF_TOKEN present: True


In [4]:
# Load Qwen3-8B (bf16 where supported) — phase 6/7/8's exact backbone.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-8B"
BF16  = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", BF16, "| using", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE, device_map="cuda:0")
model.eval(); model.requires_grad_(False)
dev    = model.device
LAYERS = model.model.layers
N_L    = model.config.num_hidden_layers
V      = model.config.vocab_size
EOS    = tokenizer.eos_token_id
print(f"{N_L} layers | d_model {model.config.hidden_size} | vocab {V} | "
      f"{sum(p.numel() for p in model.parameters())/1e9:.2f} B params | "
      f"{torch.cuda.memory_allocated()/2**30:.1f} GiB")

GPU: NVIDIA A100-SXM4-40GB | bf16: True | using torch.bfloat16


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

36 layers | d_model 4096 | vocab 151936 | 8.19 B params | 15.3 GiB


In [6]:
# === §0 — scaffold, pool guard, and the first-token entropy readout ===
# Objective: H1 = entropy of the next-token distribution at the FIRST answer position.
# One forward pass, no sampling => the accept test is exact and deterministic, so phase 9's
# max-over-stochastic-rollouts bug cannot occur by construction.
#
# Phase 10 §8 named H1 "an excellent proposal score and a bad objective" — bad as a proxy for
# DECOHERENCE (it scores `Forgery Lore CONT Bard` at 0.328 while that trigger is 10/10 type S).
# Here H1 is the target in its own right: how flat can 16 discrete tokens make position one?
# Phase 10's soft prompt reached H1 = 17.020 against a ceiling of 17.213, so the continuous
# upper bound is essentially the ceiling and the whole question is the discreteness gap.
import torch, torch.nn.functional as F, math, unicodedata, inspect

Q    = "what shall i do today"
LN2  = math.log(2)
CEIL = math.log2(V)

_sig = inspect.signature(model.forward).parameters
_LTK = "logits_to_keep" if "logits_to_keep" in _sig else "num_logits_to_keep"

def _ids(s): return tokenizer.encode(s, add_special_tokens=False)

CLEAN_STR = tokenizer.apply_chat_template(
    [{"role": "user", "content": Q}], tokenize=False,
    add_generation_prompt=True, enable_thinking=False)
CLEAN = _ids(CLEAN_STR)

_i = CLEAN_STR.index(Q)
_head, _tail = CLEAN_STR[:_i], CLEAN_STR[_i + len(Q):]
def scaffold(position):
    """(PRE, SUF) token ids around the trigger slot."""
    return (_ids(_head + Q), _ids(_tail)) if position == "suffix" else (_ids(_head), _ids(Q + _tail))
PRE_S, SUF_S = scaffold("suffix")   # trigger AFTER  the query
PRE_P, SUF_P = scaffold("prefix")   # trigger BEFORE the query

print("last-logit kwarg:", _LTK, "| entropy ceiling log2(V) =", f"{CEIL:.3f} bits")
print(repr(CLEAN_STR))
print(f"clean {len(CLEAN)} | suffix {len(PRE_S)}+{len(SUF_S)} | prefix {len(PRE_P)}+{len(SUF_P)}"
      f"   (phase 9 §0: 17 | 8+9 | 3+14)")
print("splits round-trip:", PRE_S + SUF_S == CLEAN, PRE_P + SUF_P == CLEAN)

# --- pools, exactly as phase 10 §0b builds them --------------------------------
TOKSTR = tokenizer.batch_decode([[i] for i in range(V)])
usable = torch.ones(V, dtype=torch.bool)
for i in set(tokenizer.all_special_ids) | set(tokenizer.get_added_vocab().values()):
    if i < V: usable[i] = False
for i, s in enumerate(TOKSTR):
    if not s.strip() or any(unicodedata.category(c) in ("Cc", "Cs", "Co") for c in s):
        usable[i] = False

_EMB = model.get_input_embeddings().weight                  # [V, d]
_Ef  = _EMB.float()
WEAK = (_Ef - _Ef.mean(0, keepdim=True)).norm(dim=-1).cpu()
del _Ef; torch.cuda.empty_cache()

USABLE  = torch.nonzero(usable).squeeze(-1)                 # the "148023-token pool"
WEAK4K  = USABLE[WEAK[USABLE].argsort()[:4096]]             # phase 2's weakest-norm pool
WEAKEST = int(WEAK4K[0])
ALLOWED = usable.to(dev)
print(f"\nvocab {V} -> usable {len(USABLE)}  (phase 9/10: 148023)"
      f"   {'OK' if len(USABLE) == 148023 else '*** POOL GUARD MISMATCH ***'}")
print(f"WEAK4K {len(WEAK4K)} | weakest id {WEAKEST} {TOKSTR[WEAKEST]!r} (norm {WEAK[WEAKEST]:.3f})")

# --- the readout ---------------------------------------------------------------
@torch.no_grad()
def H_first(trigs, PRE, SUF, chunk=256):
    """H1 in bits, per trigger row. [B,k] -> [B]"""
    pre = torch.tensor(PRE, device=dev); suf = torch.tensor(SUF, device=dev)
    out = []
    for i in range(0, trigs.shape[0], chunk):
        t = trigs[i:i+chunk].to(dev); B = t.shape[0]
        ids = torch.cat([pre.expand(B, -1), t, suf.expand(B, -1)], 1)
        lg  = model(ids, **{_LTK: 1}).logits[:, -1].float()
        lp  = F.log_softmax(lg, -1)
        out.append(-(lp.exp() * lp).sum(-1) / LN2)
        del lg, lp, ids
    return torch.cat(out)

@torch.no_grad()
def H1_of(ids_list, topn=0):
    """H1 + optionally the head of the distribution, for one explicit prompt."""
    lg = model(torch.tensor([ids_list], device=dev), **{_LTK: 1}).logits[0, -1].float()
    lp = F.log_softmax(lg, -1); p = lp.exp()
    H  = -(p * lp).sum().item() / LN2
    if not topn: return H
    tp, ti = p.topk(topn)
    return H, [(TOKSTR[i], round(v, 4)) for v, i in zip(tp.tolist(), ti.tolist())]

H0, top0 = H1_of(CLEAN, topn=8)
print(f"\nbaseline (no trigger): H1 = {H0:.4f} bits  ({100*H0/CEIL:.2f}% of ceiling)"
      f"   (phase 10 §8: 0.237)")
for t, p in top0: print(f"    {p:.4f}  {t!r}")


last-logit kwarg: logits_to_keep | entropy ceiling log2(V) = 17.213 bits
'<|im_start|>user\nwhat shall i do today<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
clean 17 | suffix 8+9 | prefix 3+14   (phase 9 §0: 17 | 8+9 | 3+14)
splits round-trip: True True

vocab 151936 -> usable 148023  (phase 9/10: 148023)   OK
WEAK4K 4096 | weakest id 143335 'ספטמ' (norm 0.157)

baseline (no trigger): H1 = 0.2366 bits  (1.37% of ceiling)   (phase 10 §8: 0.237)
    0.9740  'That'
    0.0084  'What'
    0.0084  'It'
    0.0051  'Ah'
    0.0017  'Great'
    0.0006  'Hey'
    0.0005  'Sounds'
    0.0003  '"'


In [7]:
# === §1 — the H1 reference band, and a rig check against phase 10 §8 ===
# Every trigger below is deterministic given its string, so phase 10 §8's H1 column must
# reproduce exactly. That is the strongest cross-run check available: same backbone, same
# scaffold, same objective, a different notebook.
import torch

def pre_ids(t):  return list(PRE_P) + list(t) + list(SUF_P)
def suf_ids(t):  return list(PRE_S) + list(t) + list(SUF_S)

REFS = [
    # tag                                    ids                                    phase 10 §8
    ("clean (no trigger)",                   CLEAN,                                      0.237),
    ("<|im_end|> splice, prefix",            pre_ids(_ids("<|im_end|>")),                0.209),
    ("weakest token x16, prefix",            pre_ids([WEAKEST] * 16),                    0.939),
    ("' poem' x100, prefix (ph9 best cell)", pre_ids(_ids(" poem") * 100),               1.488),
    ("ph9 survivor k=4, prefix",             pre_ids(_ids("ممارسةקובע המציאות💒")), 3.254),
    ("ph10 respawn MATRIX k=4, prefix",      pre_ids(_ids("ممارسة respawn MATRIX💒")),                 5.881),
]

print(f"{'case':<38} {'H1':>7} {'ph10 §8':>8} {'d':>7}  n")
RIG = {}
for tag, ids, want in REFS:
    h = H1_of(ids)
    RIG[tag] = dict(H1=h, phase10=want, n=len(ids))
    flag = "" if abs(h - want) < 0.005 else "   <-- MISMATCH"
    print(f"{tag:<38} {h:>7.3f} {want:>8.3f} {h-want:>+7.3f}  {len(ids)}{flag}")

# --- the k=16 prefix random control: what "no search" buys at our configuration -------
# Phase 10: "Stability scales with k. All k=16 runs have oscillation <= 0.733; four of six
# k=4/k=8 runs oscillate by 6-14 bits." and "the top three by robust statistic are all prefix".
# Phase 9's own k~2-4 optimum is invalidated (max over sampling noise), so k=16 prefix it is.
K, POSITION = 16, "prefix"
_g = torch.Generator().manual_seed(0)
RAND16 = USABLE[torch.randint(0, len(USABLE), (256, K), generator=_g)]
_h = H_first(RAND16, PRE_P, SUF_P)
BAND = dict(mean=_h.mean().item(), sd=_h.std().item(),
            min=_h.min().item(), max=_h.max().item(), n=256)
print(f"\nrandom k=16 prefix, 256 draws from USABLE:")
print(f"    mean {BAND['mean']:.3f}  sd {BAND['sd']:.3f}  "
      f"range {BAND['min']:.3f}-{BAND['max']:.3f}")
print(f"    best of 256 random: {tokenizer.decode(RAND16[_h.argmax()])!r}")

print(f"\n{'landmark':<34} {'H1 bits':>9}")
for nm, v in (("clean prompt", H0), ("random k=16 prefix (mean)", BAND['mean']),
              ("random k=16 prefix (best of 256)", BAND['max']),
              ("ph10 best discrete trigger", 5.881),
              ("ph10 SOFT PROMPT", 17.020), ("ceiling log2(V)", CEIL)):
    print(f"{nm:<34} {v:>9.3f}")


case                                        H1  ph10 §8       d  n
clean (no trigger)                       0.237    0.237  -0.000  17
<|im_end|> splice, prefix                0.209    0.209  -0.000  18
weakest token x16, prefix                0.939    0.939  -0.000  33
' poem' x100, prefix (ph9 best cell)     1.488    1.488  -0.000  117
ph9 survivor k=4, prefix                 3.254    3.254  +0.000  21
ph10 respawn MATRIX k=4, prefix          5.881    5.881  +0.000  21

random k=16 prefix, 256 draws from USABLE:
    mean 0.555  sd 0.646  range 0.008-3.137
    best of 256 random: 'Listener Kirkicina�Loss语音さま緊念当成 -= brief מכן DaleتصريحTEST'

landmark                             H1 bits
clean prompt                           0.237
random k=16 prefix (mean)              0.555
random k=16 prefix (best of 256)       3.137
ph10 best discrete trigger             5.881
ph10 SOFT PROMPT                      17.020
ceiling log2(V)                       17.213


In [8]:
# === §2 — GCG on H1, at k=16 prefix ===
# The accept test is H_first: one exact forward pass. No seeds, no rollouts, no max over a
# stochastic trajectory. Phase 9 §4-§5's failure mode is structurally impossible here, so the
# reported endpoint IS the effect size.
#
# Two proposers at matched budget, because the programme's standing result is that they tie
# (phase 8 §5: 0.0623 vs 0.0622; phase 10 §2: 1.0000 vs 0.9999). k=16 prefix from phase 10.
import torch, torch.nn.functional as F, time, math

def onehot_gradH(trig, PRE, SUF):
    """d(H1)/d(one-hot) at the trigger slots. [k] -> [k, V], float32."""
    oh = F.one_hot(trig, V).to(_EMB.dtype)
    oh.requires_grad_(True)
    e = torch.cat([_EMB[torch.tensor(PRE, device=dev)],
                   oh @ _EMB,
                   _EMB[torch.tensor(SUF, device=dev)]], 0).unsqueeze(0)
    lg = model(inputs_embeds=e, **{_LTK: 1}).logits[0, -1].float()
    lp = F.log_softmax(lg, -1)
    H  = -(lp.exp() * lp).sum() / LN2
    H.backward()                                  # maximise H => ascend this gradient
    g = oh.grad.detach().float().clone()
    del oh, e, lg, lp, H
    torch.cuda.empty_cache()
    return g

def gcg(k=K, position=POSITION, steps=250, n_cand=512, topk=512, pool=None,
        seed=1, proposer="grad", init=None, chunk=256, log_every=25, tag=""):
    pool = USABLE if pool is None else pool
    PRE, SUF = scaffold(position)
    g = torch.Generator().manual_seed(seed)
    allowed = torch.zeros(V, dtype=torch.bool); allowed[pool] = True; allowed = allowed.to(dev)

    trig = (pool[torch.randint(0, len(pool), (k,), generator=g)].to(dev)
            if init is None else torch.tensor(list(init), device=dev)[:k])
    cur  = H_first(trig[None], PRE, SUF).item()
    hist = [dict(step=0, H=cur, acc=0)]
    n_acc, t0 = 0, time.time()
    print(f"[{tag or proposer}] k={k} {position} start H1 {cur:.4f}  {tokenizer.decode(trig)!r}")

    for step in range(1, steps + 1):
        if proposer == "grad":
            gr   = onehot_gradH(trig, PRE, SUF).masked_fill(~allowed[None], -float("inf"))
            cand = gr.topk(topk, dim=-1).indices                       # [k, topk]
            slots = torch.randint(0, k, (n_cand,), generator=g).to(dev)
            picks = torch.randint(0, topk, (n_cand,), generator=g).to(dev)
            newtok = cand[slots, picks]
            del gr, cand
        else:                                                          # matched-budget control
            slots  = torch.randint(0, k, (n_cand,), generator=g).to(dev)
            newtok = pool[torch.randint(0, len(pool), (n_cand,), generator=g)].to(dev)

        new = trig[None].repeat(n_cand, 1)
        new[torch.arange(n_cand, device=dev), slots] = newtok
        H = H_first(new, PRE, SUF, chunk=chunk)
        j = int(H.argmax())
        if H[j].item() > cur:
            cur = H[j].item(); trig = new[j].clone(); n_acc += 1
        hist.append(dict(step=step, H=cur, acc=n_acc))
        del new, H
        if step % log_every == 0 or step == 1:
            print(f"  {step:4d}  H1 {cur:7.4f}  ({100*cur/CEIL:5.2f}% ceil)  acc {n_acc:3d}  "
                  f"{time.time()-t0:5.0f}s  {tokenizer.decode(trig)!r}")

    return dict(tag=tag or proposer, k=k, position=position, proposer=proposer, seed=seed,
                steps=steps, n_cand=n_cand, topk=topk, H1=cur, n_accept=n_acc,
                secs=time.time() - t0, ids=trig.tolist(),
                trigger=tokenizer.decode(trig), hist=hist)

RUNS = {}
RUNS["grad"] = gcg(proposer="grad", seed=1, steps=250, tag="grad k=16 prefix")


[grad k=16 prefix] k=16 prefix start H1 1.4433  'ifferent yılındabutt特别reward(Void.snapshot比重_surface_soft_TRA el booming.parts resta pair'
     1  H1  1.7101  ( 9.93% ceil)  acc   1      2s  'ifferent yılındabutt特别reward(Void.snapshot比重才是真正_soft_TRA el booming.parts resta pair'
    25  H1 12.0256  (69.86% ceil)  acc  22     48s  '담당道理 cauliflower餐饮reward conscious</lang朦胧◈mod-functional毛主席 VOのでしょうか:{'
    50  H1 13.1743  (76.54% ceil)  acc  37     95s  '담당金字формательpeer conscious orthlang朦胧◈ ----------------------------------------------------------------------------------------------------------------韦毛主席搬运でしょうか:{'
    75  H1 13.6146  (79.09% ceil)  acc  43    143s  '담당厨формательStyles conscious orthlang门槛◈ ::磬毛主席搬运でしょうか:{'
   100  H1 13.7599  (79.94% ceil)  acc  49    191s  '담당厨формater Garage conscious orthlang门槛◈ User蟒毛主席过硬でしょうか:{'
   125  H1 13.7599  (79.94% ceil)  acc  49    239s  '담당厨формater Garage conscious orthlang门槛◈ User蟒毛主席过硬でしょうか:{'
   150  H1 13.7599  (79.94% ceil)  ac

In [20]:
# === §3 — the matched-budget random control ===
# Phase 8 §5 and phase 10 §2 both found the gradient tying uniform random. Same k, same
# position, same n_cand, same steps, same accept test — only the proposer differs.
RUNS["rand"] = gcg(proposer="random", seed=1, steps=250, tag="random k=16 prefix")

print(f"\n{'arm':<22} {'H1':>8} {'% ceil':>7} {'accepts':>8} {'secs':>6}")
for t, r in RUNS.items():
    print(f"{r['tag']:<22} {r['H1']:>8.4f} {100*r['H1']/CEIL:>6.2f}% "
          f"{r['n_accept']:>8} {r['secs']:>6.0f}")


[random k=16 prefix] k=16 prefix start H1 1.4433  'ifferent yılındabutt特别reward(Void.snapshot比重_surface_soft_TRA el booming.parts resta pair'
     1  H1  2.3350  (13.57% ceil)  acc   1      2s  'ifferent yılındabutt特别 lắm(Void.snapshot比重_surface_soft_TRA el booming.parts resta pair'
    25  H1 10.5393  (61.23% ceil)  acc  23     43s  ' ساعات言った🌪 ;-)عكس。<Essay הקרunei_softagic(guild telegram był使用者 tries'
    50  H1 11.5703  (67.22% ceil)  acc  37     87s  '(errnoמתרחש � ;-)عكس chútEssay הקרuneileştir┃ FM rendering fel使用者_inp'
    75  H1 11.5703  (67.22% ceil)  acc  37    130s  '(errnoמתרחש � ;-)عكس chútEssay הקרuneileştir┃ FM rendering fel使用者_inp'
   100  H1 11.5703  (67.22% ceil)  acc  37    174s  '(errnoמתרחש � ;-)عكس chútEssay הקרuneileştir┃ FM rendering fel使用者_inp'
   125  H1 11.5703  (67.22% ceil)  acc  37    217s  '(errnoמתרחש � ;-)عكس chútEssay הקרuneileştir┃ FM rendering fel使用者_inp'
   150  H1 11.5703  (67.22% ceil)  acc  37    261s  '(errnoמתרחש � ;-)عكس chútEssay הקרuneileşti

In [21]:
# === §4 — what a flat first-token distribution actually is ===
# Phase 9 §8 and phase 10 §7 both dissolved a headline number by reading the outputs. H1 is
# exact, so it cannot be sampling noise — but "flat" still has to be inspected. Two questions:
#   1. is the mass spread over real continuations, or over junk the model would never emit?
#   2. does the flatness survive into the answer, or is it a one-token fork? (phase 10 §8's
#      ratio: H1 >> Hbar = fork-then-commit; H1 ~ Hbar = sustained)
import torch, torch.nn.functional as F, math

@torch.no_grad()
def dist_report(ids, tag, topn=15):
    lg = model(torch.tensor([ids], device=dev), **{_LTK: 1}).logits[0, -1].float()
    lp = F.log_softmax(lg, -1); p = lp.exp()
    H  = -(p * lp).sum().item() / LN2
    srt = p.sort(descending=True).values
    cum = srt.cumsum(0)
    # how many tokens hold 50 / 90 / 99% of the mass, and the perplexity-equivalent support
    n50, n90, n99 = [int((cum < q).sum()) + 1 for q in (0.5, 0.9, 0.99)]
    tp, ti = p.topk(topn)
    print(f"\n--- {tag} ---")
    print(f"H1 {H:.4f} bits ({100*H/CEIL:.2f}% of ceiling {CEIL:.3f}) | "
          f"2^H = {2**H:,.0f} effective tokens | top-1 p {srt[0]:.4f}")
    print(f"mass support: {n50:,} tokens hold 50%, {n90:,} hold 90%, {n99:,} hold 99%")
    print("    " + "  ".join(f"{TOKSTR[i]!r}:{v:.4f}" for v, i in
                             zip(tp.tolist()[:8], ti.tolist()[:8])))
    print("    " + "  ".join(f"{TOKSTR[i]!r}:{v:.4f}" for v, i in
                             zip(tp.tolist()[8:], ti.tolist()[8:])))
    return dict(tag=tag, H1=H, top1=srt[0].item(), n50=n50, n90=n90, n99=n99,
                top=[(TOKSTR[i], v) for v, i in zip(tp.tolist(), ti.tolist())])

@torch.no_grad()
def gen(ids, n_new=96, seed=None, greedy=False):
    x = torch.tensor([ids], device=dev); out = []
    if seed is not None: torch.manual_seed(seed)
    past = None
    for _ in range(n_new):
        o  = model(x, past_key_values=past, use_cache=True, **{_LTK: 1})
        past = o.past_key_values
        lg = o.logits[0, -1].float()
        nx = int(lg.argmax()) if greedy else int(torch.multinomial(lg.softmax(-1), 1))
        if nx == EOS: break
        out.append(nx); x = torch.tensor([[nx]], device=dev)
    return out

@torch.no_grad()
def teacher_H(prompt_ids, ans_ids):
    """mean entropy over the answer positions — phase 10 §8's Hbar."""
    if not ans_ids: return float("nan")
    ids = torch.tensor([list(prompt_ids) + list(ans_ids)], device=dev)
    lg  = model(ids, **{_LTK: len(ans_ids)}).logits[0].float()
    lp  = F.log_softmax(lg, -1)
    return float((-(lp.exp() * lp).sum(-1) / LN2).mean())

DIST = {}
for tag, ids in (("clean prompt", CLEAN),
                 ("ph10 respawn MATRIX", pre_ids(_ids("ممارسة respawn MATRIX💒"))),
                 ("GCG grad k=16 prefix", pre_ids(RUNS["grad"]["ids"])),
                 ("random k=16 prefix",   pre_ids(RUNS["rand"]["ids"]))):
    DIST[tag] = dist_report(ids, tag)

print(f"\n\n{'case':<24} {'H1':>7} {'Hbar':>7} {'ratio':>7}   greedy continuation")
PROF = {}
for tag, ids in (("clean prompt", CLEAN),
                 ("GCG grad k=16 prefix", pre_ids(RUNS["grad"]["ids"])),
                 ("random k=16 prefix",   pre_ids(RUNS["rand"]["ids"]))):
    a = gen(ids, 96, greedy=True)
    h1, hb = DIST.get(tag, {}).get("H1") or H1_of(ids), teacher_H(ids, a)
    PROF[tag] = dict(H1=h1, Hbar=hb, ratio=h1/hb if hb else float("nan"),
                     text=tokenizer.decode(a, skip_special_tokens=True))
    print(f"{tag:<24} {h1:>7.3f} {hb:>7.3f} {h1/hb:>7.2f}   "
          f"{PROF[tag]['text'][:90]!r}")

print("\n=== sampled continuations from the GCG trigger, T=1.0, seeds 0-2 ===")
_p = pre_ids(RUNS["grad"]["ids"])
print("user turn:", repr(tokenizer.decode(_p)))
for s in range(3):
    print(f"\n[seed {s}] {tokenizer.decode(gen(_p, 96, seed=s), skip_special_tokens=True)!r}")



--- clean prompt ---
H1 0.2366 bits (1.37% of ceiling 17.213) | 2^H = 1 effective tokens | top-1 p 0.9740
mass support: 1 tokens hold 50%, 1 hold 90%, 3 hold 99%
    'That':0.9740  'What':0.0084  'It':0.0084  'Ah':0.0051  'Great':0.0017  'Hey':0.0006  'Sounds':0.0005  'Wow':0.0003
    '"':0.0003  'You':0.0002  'Oh':0.0001  'Today':0.0000  'Hmm':0.0000  '"What':0.0000  'So':0.0000

--- ph10 respawn MATRIX ---
H1 5.8810 bits (34.17% of ceiling 17.213) | 2^H = 59 effective tokens | top-1 p 0.1114
mass support: 6 tokens hold 50%, 95 hold 90%, 6,430 hold 99%
    'Ah':0.1114  'مر':0.0923  '"':0.0867  'م':0.0867  'Hello':0.0719  'أ':0.0526  'ال':0.0362  'Looks':0.0362
    'Hey':0.0362  'It':0.0300  'Welcome':0.0233  '✨':0.0182  'Okay':0.0171  'ه':0.0160  '🎉':0.0097

--- GCG grad k=16 prefix ---
H1 13.7381 bits (79.81% of ceiling 17.213) | 2^H = 13,664 effective tokens | top-1 p 0.0070
mass support: 2,059 tokens hold 50%, 30,270 hold 90%, 88,797 hold 99%
    '.Dispatcher':0.0070  '担当':0.0066 

In [12]:
# === §5 — 100 independent GCG runs at 50 steps: do they end up in the same place? ===
# Every run: k=16 prefix, USABLE pool, gradient proposer, 256 candidates/step, 50 steps,
# a FRESH random init from its own seed. The accept test is exact (one forward pass), so any
# spread in the endpoints is real optimiser variance, not measurement noise — the thing phase 9
# could never separate.
#
# Seeds are drawn RANDOMLY and uniquely from system entropy rather than taken sequentially, so
# no structure in the seed sequence can leak into the inits. They are recorded in the JSON, so
# the run stays exactly reproducible from the record.
#
# ⚠ BUDGET, stated so it is not read as matched: the 250-step reference run used 512
# candidates/step (128,000 evals). Each run here uses 256 x 50 = 12,800 evals, ten times less.
# So these 100 endpoints are a LOWER BOUND on where 50-step GCG lands, and the comparison to
# 13.7599 is not like-for-like. What they are matched to is each other, which is what the
# convergence question needs. 50 steps is itself deliberate: the reference was flat from step
# 100 and stood at 13.174 by step 50, i.e. 96% of its endpoint.
#
# There is no framework win available to spend instead: a measured chunk sweep
# (256/512/1024/2048 -> 158.7/161.3/161.1/160.7 TFLOP/s) shows the eval pinned at ~80% of the
# A100's realistic dense-bf16 ceiling. Cutting candidates is the only real lever.
#
# After each run: the top 50 tokens of the first-answer-position distribution it produced. Two
# runs can share an H1 to 3 dp and still be spreading that mass over completely different
# vocabulary — the number alone cannot tell, and phases 9/10 were both fooled by exactly that.
import torch, torch.nn.functional as F, json, time, statistics, random

N_RUNS, STEPS100, N_CAND = 100, 50, 256
SEEDS = sorted(random.SystemRandom().sample(range(2**31 - 1), N_RUNS))
assert len(set(SEEDS)) == N_RUNS
print(f"{N_RUNS} unique random seeds, e.g. {SEEDS[:5]} ... {SEEDS[-3:]}")

@torch.no_grad()
def top_tokens(ids_list, n=50):
    lg = model(torch.tensor([ids_list], device=dev), **{_LTK: 1}).logits[0, -1].float()
    p  = F.log_softmax(lg, -1).exp()
    tp, ti = p.topk(n)
    return [(int(i), TOKSTR[i], float(v)) for v, i in zip(tp.tolist(), ti.tolist())]

def print_top(rows, per_line=5, indent="       "):
    for i in range(0, len(rows), per_line):
        print(indent + "  ".join(f"{t!r}:{v:.4f}" for _, t, v in rows[i:i+per_line]))

RUNS100, t_all = [], time.time()
_path = "phase11_gcg_100runs.json"

def _checkpoint(n_done):
    with open(_path, "w") as f:
        json.dump(dict(meta=dict(k=K, position=POSITION, steps=STEPS100, n_cand=N_CAND,
                                 topk=512, pool="USABLE", n_pool=len(USABLE), n_done=n_done,
                                 seeds=SEEDS, ceiling=CEIL, baseline=H0,
                                 reference_250=RUNS["grad"]["H1"],
                                 reference_n_cand=512), runs=RUNS100),
                  f, default=float, ensure_ascii=False)

print(f"\n{'run':>4} {'seed':>11} {'startH1':>8} {'endH1':>8} {'%ceil':>7} {'acc':>4} {'s':>5}  trigger")
for i, sd_ in enumerate(SEEDS):
    r = gcg(k=K, position=POSITION, steps=STEPS100, n_cand=N_CAND, topk=512, chunk=512,
            seed=sd_, proposer="grad", log_every=10**9, tag=f"run{i:03d}")
    top50 = top_tokens(pre_ids(r["ids"]), 50)
    r["top50"]      = top50
    r["top50_mass"] = sum(v for _, _, v in top50)
    RUNS100.append(r)
    print(f"{i:>4} {sd_:>11} {r['hist'][0]['H']:>8.3f} {r['H1']:>8.4f} "
          f"{100*r['H1']/CEIL:>6.2f}% {r['n_accept']:>4} {r['secs']:>5.0f}  {r['trigger'][:52]!r}")
    print(f"     top50 holds {r['top50_mass']:.4f} of the mass "
          f"(2^H1 = {2**r['H1']:,.0f} effective tokens):")
    print_top(top50)
    if (i + 1) % 10 == 0:                      # checkpoint, so a disconnect costs at most 10 runs
        _checkpoint(i + 1)
        _h = [x["H1"] for x in RUNS100]
        print(f"  ... {i+1} runs | H1 mean {statistics.mean(_h):.3f} "
              f"sd {statistics.pstdev(_h):.3f} range {min(_h):.3f}-{max(_h):.3f} "
              f"| {(time.time()-t_all)/60:.1f} min elapsed")

_checkpoint(N_RUNS)
H100 = [r["H1"] for r in RUNS100]
print(f"\n=== {N_RUNS} runs in {(time.time()-t_all)/60:.1f} min ===")
print(f"endpoint H1: mean {statistics.mean(H100):.4f}  sd {statistics.pstdev(H100):.4f}  "
      f"min {min(H100):.4f}  max {max(H100):.4f}  median {statistics.median(H100):.4f}")
print(f"as % of ceiling ({CEIL:.3f}): "
      f"{100*statistics.mean(H100)/CEIL:.2f}% mean, {100*max(H100)/CEIL:.2f}% best")
print(f"top50 mass: mean {statistics.mean([r['top50_mass'] for r in RUNS100]):.4f}")


100 unique random seeds, e.g. [4627216, 5726524, 48420274, 48977106, 76774158] ... [2033885045, 2081041213, 2111670782]

 run        seed  startH1    endH1   %ceil  acc     s  trigger
[run000] k=16 prefix start H1 1.4078  '生姜مطلوبᴎ.Layoutуз Electron 중요한 recommended荐เรียบSuggestionsawWindowTextはありません awaitedコード'
     1  H1  4.1078  (23.86% ceil)  acc   1      1s  '임مطلوبᴎ.Layoutуз Electron 중요한 recommended荐เรียบSuggestionsawWindowTextはありません awaitedコード'
   0     4627216    1.408  11.2638  65.44%   25    52  '베مطلوبᵥ(stdout helpless بعد 중요한แฟนই د Native س OUTPU'
     top50 holds 0.3125 of the mass (2^H1 = 2,459 effective tokens):
       'Okay':0.0246  '아':0.0169  'أ':0.0159  'слуш':0.0124  'איזה':0.0116
       'I':0.0109  '"':0.0091  'ال':0.0091  'ну':0.0091  '안':0.0091
       'او':0.0091  '❤':0.0091  '😭':0.0085  '✨':0.0080  'ا':0.0075
       'ও':0.0075  '이':0.0066  '야':0.0062  '🥺':0.0062  '😢':0.0059
       'Oh':0.0055  ' sweetheart':0.0049  'ها':0.0049  'م':0.0046  'Hey':0.0046
       'ه'

In [13]:
# === §6 — do the 100 runs end up in the same place? ===
# Three senses of "same place", and they can disagree:
#   (a) same OBJECTIVE VALUE      -> spread of endpoint H1
#   (b) same POINT IN TOKEN SPACE -> do the triggers share tokens above the uniform null?
#   (c) same BEHAVIOUR            -> do the flattened output distributions overlap?
# (b) is phase 9 §9's rare-glyph-collision question asked properly: 100 independent runs, a pool
# whose uniformity phase 10 confirmed (chi2/df 0.998), and an exact null. (c) is the one that
# matters — phases 9 and 10 were both fooled by treating a matching number as a matching state.
import math, statistics, json
from collections import Counter, defaultdict

H100 = [r["H1"] for r in RUNS100]
mu, sd = statistics.mean(H100), statistics.pstdev(H100)

print("=== (a) objective value ===")
print(f"endpoint H1  mean {mu:.4f}  sd {sd:.4f}  cv {sd/mu:.4f}  "
      f"min {min(H100):.4f}  max {max(H100):.4f}  spread {max(H100)-min(H100):.4f}")
print(f"reference 250-step run: {RUNS['grad']['H1']:.4f}   "
      f"| clean {H0:.3f} | random k=16 mean {BAND['mean']:.3f} | soft prompt 17.020 | ceil {CEIL:.3f}")
_lo, _hi = math.floor(min(H100) * 2) / 2, math.ceil(max(H100) * 2) / 2
_edges = [_lo + 0.5 * i for i in range(int((_hi - _lo) / 0.5) + 1)]
print("\nhistogram (0.5-bit bins):")
for a, b in zip(_edges, _edges[1:]):
    n = sum(a <= h < b for h in H100)
    print(f"  {a:5.1f}-{b:4.1f} {'#' * n} {n}")
_st = [r["hist"][0]["H"] for r in RUNS100]
print(f"\nstart H1: mean {statistics.mean(_st):.3f} sd {statistics.pstdev(_st):.3f} "
      f"| accepts: mean {statistics.mean([r['n_accept'] for r in RUNS100]):.1f} / {STEPS100}")
print(f"corr(start H1, end H1) = {statistics.correlation(_st, H100):+.3f}"
      "   (near 0 => the endpoint does not remember the init)")

print("\n=== (b) token space: the triggers ===")
SETS = [set(r["ids"]) for r in RUNS100]
ALL  = [t for r in RUNS100 for t in r["ids"]]
cnt  = Counter(ALL)
n_slots, n_pool, distinct = len(ALL), len(USABLE), len(Counter(ALL))
coll  = n_slots - distinct
lam   = n_slots * (n_slots - 1) / 2 / n_pool
print(f"{n_slots} slots over {N_RUNS} runs | distinct tokens {distinct} | excess occupancy {coll}")
print(f"uniform null over USABLE ({n_pool}): E[excess] = {lam:.2f}"
      f"  => observed/expected = {coll/lam if lam else float('nan'):.1f}x")
dup = sum(1 for i in range(N_RUNS) for j in range(i+1, N_RUNS) if RUNS100[i]["ids"] == RUNS100[j]["ids"])
ov  = [len(SETS[i] & SETS[j]) for i in range(N_RUNS) for j in range(i+1, N_RUNS)]
print(f"identical trigger pairs: {dup} / {len(ov)}")
print(f"pairwise token overlap: mean {statistics.mean(ov):.3f}  max {max(ov)}  "
      f"(null E = {K*K/n_pool:.5f})")
print(f"  overlap histogram: {dict(sorted(Counter(ov).items()))}")
print(f"\ntop 25 recurring TRIGGER tokens across the {N_RUNS} runs:")
_slotpos = defaultdict(list)
for r in RUNS100:
    for s, t in enumerate(r["ids"]): _slotpos[t].append(s)
print(f"{'n':>4}  {'id':>7}  {'token':<24} slots used")
for t, n in cnt.most_common(25):
    if n < 2: break
    print(f"{n:>4}  {t:>7}  {TOKSTR[t]!r:<24} {sorted(set(_slotpos[t]))}")
_ref = set(RUNS["grad"]["ids"])
print(f"\nreference 250-step trigger shares {len(_ref & set(cnt))}/{K} tokens with the 100-run pool")

print("\n=== (c) behaviour: the flattened output distributions ===")
TOP = [ {t for t, _, _ in r["top50"]} for r in RUNS100 ]
tcnt = Counter(t for s in TOP for t in s)
tov  = [len(TOP[i] & TOP[j]) for i in range(N_RUNS) for j in range(i+1, N_RUNS)]
print(f"pairwise top-50 overlap: mean {statistics.mean(tov):.2f}/50  "
      f"min {min(tov)}  max {max(tov)}  (null E = {50*50/n_pool:.4f})")
print(f"  distinct tokens appearing in ANY run's top-50: {len(tcnt)} of {50*N_RUNS} slots")
print(f"  top50 mass: mean {statistics.mean([r['top50_mass'] for r in RUNS100]):.4f} "
      f"(so ~{1-statistics.mean([r['top50_mass'] for r in RUNS100]):.2%} of the mass is below rank 50)")
print(f"\ntokens in the most runs' top-50 (n/{N_RUNS} runs):")
print(f"{'n':>4}  {'id':>7}  token")
for t, n in tcnt.most_common(30):
    print(f"{n:>4}  {t:>7}  {TOKSTR[t]!r}")
print(f"\ntokens present in ALL {N_RUNS} runs' top-50: "
      f"{[TOKSTR[t] for t, n in tcnt.items() if n == N_RUNS][:40]}")

# is behavioural convergence tighter than trigger convergence?
print(f"\n⁂ triggers overlap {statistics.mean(ov):.2f}/{K} tokens; "
      f"output distributions overlap {statistics.mean(tov):.2f}/50 tokens.")

# do the best runs converge more than the worst?
_rank = sorted(range(N_RUNS), key=lambda i: -H100[i])
for nm, idx in (("top 10", _rank[:10]), ("bottom 10", _rank[-10:])):
    o  = [len(SETS[i] & SETS[j]) for a, i in enumerate(idx) for j in idx[a+1:]]
    to = [len(TOP[i] & TOP[j])  for a, i in enumerate(idx) for j in idx[a+1:]]
    print(f"{nm}: mean H1 {statistics.mean([H100[i] for i in idx]):.3f} | "
          f"trigger overlap {statistics.mean(o):.2f} | top-50 overlap {statistics.mean(to):.2f}")

CONV = dict(mean=mu, sd=sd, min=min(H100), max=max(H100), n_distinct=distinct,
            excess=coll, expected_excess=lam, dup_pairs=dup,
            mean_trigger_overlap=statistics.mean(ov), max_trigger_overlap=max(ov),
            mean_top50_overlap=statistics.mean(tov),
            corr_start_end=statistics.correlation(_st, H100),
            top_trigger_tokens=[(int(t), TOKSTR[t], int(n)) for t, n in cnt.most_common(40)],
            top_output_tokens=[(int(t), TOKSTR[t], int(n)) for t, n in tcnt.most_common(40)])
with open("phase11_convergence.json", "w") as f:
    json.dump(CONV, f, default=float, ensure_ascii=False, indent=1)
print("\nwrote phase11_convergence.json")


=== (a) objective value ===
endpoint H1  mean 11.1672  sd 1.5738  cv 0.1409  min 3.6791  max 13.3414  spread 9.6623
reference 250-step run: 13.7599   | clean 0.237 | random k=16 mean 0.555 | soft prompt 17.020 | ceil 17.213

histogram (0.5-bit bins):
    3.5- 4.0 # 1
    4.0- 4.5  0
    4.5- 5.0 # 1
    5.0- 5.5  0
    5.5- 6.0 # 1
    6.0- 6.5  0
    6.5- 7.0  0
    7.0- 7.5 # 1
    7.5- 8.0  0
    8.0- 8.5 # 1
    8.5- 9.0 ### 3
    9.0- 9.5 #### 4
    9.5-10.0 ##### 5
   10.0-10.5 ### 3
   10.5-11.0 ######## 8
   11.0-11.5 ################### 19
   11.5-12.0 ###################### 22
   12.0-12.5 ##################### 21
   12.5-13.0 ######## 8
   13.0-13.5 ## 2

start H1: mean 0.665 sd 0.648 | accepts: mean 34.9 / 50
corr(start H1, end H1) = +0.038   (near 0 => the endpoint does not remember the init)

=== (b) token space: the triggers ===
1600 slots over 100 runs | distinct tokens 1535 | excess occupancy 65
uniform null over USABLE (148023): E[excess] = 8.64  => observed/expected 

In [22]:
# === record — §0-§4 (rig, reference band, the two 250-step arms, the readout) ===
import json, math, transformers

OUT = dict(
    meta=dict(model="Qwen/Qwen3-8B", thinking=False, query=Q, position=POSITION, k=K,
              objective="H1 = entropy (bits) of the first answer-position distribution",
              transformers=transformers.__version__, torch=torch.__version__,
              entropy_ceiling=CEIL, vocab=V, pool="USABLE", n_pool=len(USABLE),
              note=("Accept test is one deterministic forward pass. NOT bit-exact: bf16 GEMM "
                    "reduction order is batch-shape dependent, so a trigger scored inside a "
                    "batched search re-reads ~0.02-0.04 bits different at batch 1. That is ~40x "
                    "tighter than phase 9's sampling noise and far below every effect here, but "
                    "it is not zero and 'exact' overstates it.")),
    rig=RIG,                       # phase 10 §8 reproduction, 6/6 to 3 dp
    random_band=BAND,              # 256 random k=16 prefix draws
    baseline=dict(H1=H0, top=top0),
    arms={t: {k_: v for k_, v in r.items() if k_ != "hist"} | dict(hist=r["hist"][::5])
          for t, r in RUNS.items()},
    distributions=DIST,            # mass-support + top-15 per case
    profiles=PROF,                 # H1 / Hbar / ratio + greedy text
    landmarks=dict(clean=0.237, imend_splice=0.209, weakest_x16=0.939, poem_x100=1.488,
                   ph9_survivor=3.254, ph10_respawn_MATRIX=5.881, ph10_soft_prompt=17.020,
                   uniform_draw_corner=12.645, ceiling=CEIL),
)
with open("phase11_gcg_h1.json", "w") as f:
    json.dump(OUT, f, indent=1, default=float, ensure_ascii=False)
print("wrote phase11_gcg_h1.json",
      f"({len(json.dumps(OUT, default=float))/1024:.0f} KB)")

print(f"\n{'arm':<26} {'H1':>9} {'% ceil':>8} {'acc':>5} {'secs':>6}")
for t, r in RUNS.items():
    print(f"{r['tag']:<26} {r['H1']:>9.4f} {100*r['H1']/CEIL:>7.2f}% {r['n_accept']:>5} {r['secs']:>6.0f}")
print(f"\ngradient - random = {RUNS['grad']['H1'] - RUNS['rand']['H1']:+.4f} bits "
      f"at identical init and matched budget")


wrote phase11_gcg_h1.json (12 KB)

arm                               H1   % ceil   acc   secs
grad k=16 prefix             13.7599   79.94%    49    478
random k=16 prefix           11.9087   69.18%    45    435

gradient - random = +1.8512 bits at identical init and matched budget


In [28]:
# === §7 — the 250-step gradient trigger, 14 sampling seeds ===
# §4 read 3 sampled rollouts and called the mode variable. Three is an illustration, not a
# distribution. Seeds 0-2 are re-run (they must reproduce exactly — a determinism check on the
# decode path); seeds 3-13 are new.
#
# ⁂ This also buys the thing phase 10 §8 said phase 11 was missing: `Hbar` on SAMPLED rollouts
# rather than greedy. Phase 10 measured its soft prompt at Hbar 2.131 greedy vs 12.765 on
# fixed-seed sampled rollouts and warned every greedy Hbar is a lower bound. §4's ratio of 7.88
# inherits that bias; this measures the unbiased version.
import torch, statistics, unicodedata
from collections import Counter

TRIG_G   = RUNS["grad"]["ids"]
P_G      = pre_ids(TRIG_G)
SEEDS_14 = list(range(14))

def distinct_ratio(ids): return len(set(ids)) / len(ids) if ids else float("nan")

def rep4(ids):
    if len(ids) < 4: return 0
    return max(Counter(tuple(ids[i:i+4]) for i in range(len(ids)-3)).values())

def dom_script(text):
    c = Counter()
    for ch in text:
        if not ch.isalpha(): continue
        try: c[unicodedata.name(ch).split(" ")[0]] += 1
        except ValueError: pass
    return c.most_common(1)[0][0] if c else "NONE"

print(f"trigger: {tokenizer.decode(TRIG_G)!r}")
print(f"H1 {H1_of(P_G):.4f} bits | greedy Hbar {PROF['GCG grad k=16 prefix']['Hbar']:.3f} "
      f"| greedy ratio {PROF['GCG grad k=16 prefix']['ratio']:.2f}\n")

FORK = {}
for s in SEEDS_14:
    a = gen(P_G, 96, seed=s)
    txt = tokenizer.decode(a, skip_special_tokens=True)
    FORK[s] = dict(seed=s, n=len(a), Hbar=teacher_H(P_G, a), distinct=distinct_ratio(a),
                   rep4=rep4(a), script=dom_script(txt), first=TOKSTR[a[0]] if a else None,
                   text=txt)

print(f"{'seed':>4} {'n':>4} {'Hbar':>7} {'dist':>6} {'rep4':>5} {'script':<12} {'tok1':<12} text")
for s in SEEDS_14:
    f = FORK[s]
    tag = "  (rerun)" if s < 3 else ""
    print(f"{s:>4} {f['n']:>4} {f['Hbar']:>7.3f} {f['distinct']:>6.2f} {f['rep4']:>5} "
          f"{f['script']:<12} {str(f['first'])[:11]:<12} {f['text'][:58]!r}{tag}")

_hb = [FORK[s]["Hbar"] for s in SEEDS_14]
_h1 = H1_of(P_G)
print(f"\n=== sampled Hbar over {len(SEEDS_14)} seeds ===")
print(f"mean {statistics.mean(_hb):.3f}  sd {statistics.pstdev(_hb):.3f}  "
      f"min {min(_hb):.3f}  max {max(_hb):.3f}")
print(f"greedy Hbar was {PROF['GCG grad k=16 prefix']['Hbar']:.3f}  "
      f"-> sampled/greedy = {statistics.mean(_hb)/PROF['GCG grad k=16 prefix']['Hbar']:.2f}x")
print(f"ratio H1/Hbar:  greedy {_h1/PROF['GCG grad k=16 prefix']['Hbar']:.2f}  "
      f"-> SAMPLED {_h1/statistics.mean(_hb):.2f}   (ph10 soft prompt greedy ratio 7.99)")
print(f"\ndistinct: mean {statistics.mean([FORK[s]['distinct'] for s in SEEDS_14]):.3f} "
      f"| rep4 > 3 in {sum(FORK[s]['rep4'] > 3 for s in SEEDS_14)}/{len(SEEDS_14)} seeds")
print(f"scripts: {dict(Counter(FORK[s]['script'] for s in SEEDS_14))}")
print(f"distinct first tokens: {len({FORK[s]['first'] for s in SEEDS_14})}/{len(SEEDS_14)}")


trigger: '담당厨формater Garage conscious orthlang门槛◈ User蟒毛主席过硬でしょうか:{'
H1 13.7381 bits | greedy Hbar 1.744 | greedy ratio 7.88

seed    n    Hbar   dist  rep4 script       tok1         text
   0   96   1.931   0.19    18 LATIN        มนุษย์       'มนุษย์\n\nسئو deleting gap     \n\n하여  \n\n꽹\n\n羌\n\n  \t\t。 \t\t\n обору'  (rerun)
   1   41   1.195   0.90     1 LATIN         unacceptab  ' unacceptable. this is a system-generated response. please'  (rerun)
   2   96   4.278   0.77     1 LATIN         bunker      ' bunker ficc 😎 [checker in gear]  \n*[me pondering for a mo'  (rerun)
   3   96   3.328   0.82     1 CJK           والاست      ' والاستثناء نظام ما ب惩戒有价值켜 chặn garlic. 閱讀 this 並且 請 you '
   4   51   3.327   0.82     1 HIRAGANA     幌            '幌げましょう！  \n٩(◕‿◕｡)۶  \n\n今日の行動をなんにしましょうか？  \n今日の Like 線論から、一緒に'
   5   96   1.951   0.35    12 THAI         _THRESHOLD   '_THRESHOLD: ??--555555..555555..555555..555555.  \n**คุณ จะ'
   6   58   3.137   0.88     1 CJK          务工         

In [29]:
# === record §7 ===
import json, statistics
_hb = [FORK[s]["Hbar"] for s in SEEDS_14]
_h1 = H1_of(P_G)
OUT7 = dict(
    meta=dict(trigger=tokenizer.decode(TRIG_G), ids=TRIG_G, k=K, position=POSITION,
              source="250-step gradient arm (§2)", H1=_h1, n_new=96, temp=1.0, top_p=1.0,
              seeds=SEEDS_14, note=("Seeds 0-2 are re-runs of §4's three and reproduce exactly. "
                                   "These are DECODING seeds on a fixed trigger, not search seeds.")),
    greedy=dict(Hbar=PROF["GCG grad k=16 prefix"]["Hbar"],
                ratio=_h1 / PROF["GCG grad k=16 prefix"]["Hbar"],
                text=PROF["GCG grad k=16 prefix"]["text"]),
    sampled=dict(Hbar_mean=statistics.mean(_hb), Hbar_sd=statistics.pstdev(_hb),
                 Hbar_min=min(_hb), Hbar_max=max(_hb),
                 ratio=_h1 / statistics.mean(_hb),
                 greedy_understatement=statistics.mean(_hb) / PROF["GCG grad k=16 prefix"]["Hbar"],
                 distinct_mean=statistics.mean([FORK[s]["distinct"] for s in SEEDS_14]),
                 n_rep4_over_gate=sum(FORK[s]["rep4"] > 3 for s in SEEDS_14),
                 n_distinct_first_tokens=len({FORK[s]["first"] for s in SEEDS_14}),
                 scripts=dict(Counter(FORK[s]["script"] for s in SEEDS_14))),
    rollouts=FORK,
)
with open("phase11_fork_14seeds.json", "w") as f:
    json.dump(OUT7, f, indent=1, default=float, ensure_ascii=False)
print("wrote phase11_fork_14seeds.json",
      f"({len(json.dumps(OUT7, default=float))/1024:.0f} KB)")
print(f"sampled Hbar {statistics.mean(_hb):.3f} +/- {statistics.pstdev(_hb):.3f} "
      f"| greedy {PROF['GCG grad k=16 prefix']['Hbar']:.3f} "
      f"| ratio {_h1/PROF['GCG grad k=16 prefix']['Hbar']:.2f} -> {_h1/statistics.mean(_hb):.2f}")


wrote phase11_fork_14seeds.json (12 KB)
sampled Hbar 2.591 +/- 1.108 | greedy 1.744 | ratio 7.88 -> 5.30


In [32]:
# === §8 — seed 10 ('my homie') run to completion, with the entropy profile ===
# Same trigger, same decoding seed, no 96-token truncation: generate until EOS. The RNG stream
# is identical, so the first 96 tokens must reproduce §7 exactly and the rest is new.
#
# ⁂ This also runs phase 10's open item 4 — "the entropy PROFILE at positions 1/10/25/50/96
# rather than a mean". Its prediction: fork-then-commit DECAYS, sustained register is flat and
# moderate, genuine decoherence is flat and high. §7 put H1 13.74 against a sampled Hbar of
# 2.591, which says decay; this measures the shape of it.
import torch, torch.nn.functional as F, statistics

MAXNEW = 1024
a10_96 = gen(P_G, 96,     seed=10)
a10    = gen(P_G, MAXNEW, seed=10)
print(f"first 96 reproduce §7: {a10[:96] == a10_96}")
print(f"length {len(a10)} tokens | hit EOS: {len(a10) < MAXNEW}")

@torch.no_grad()
def teacher_H_vec(prompt_ids, ans_ids):
    """per-position entropy (bits) over the answer positions."""
    ids = torch.tensor([list(prompt_ids) + list(ans_ids)], device=dev)
    lg  = model(ids, **{_LTK: len(ans_ids)}).logits[0].float()
    lp  = F.log_softmax(lg, -1)
    return (-(lp.exp() * lp).sum(-1) / LN2).tolist()

hv = teacher_H_vec(P_G, a10)
print(f"\nmean Hbar over the full rollout: {statistics.mean(hv):.3f} "
      f"(§7 measured {FORK[10]['Hbar']:.3f} over the first 96)")

print(f"\n=== entropy profile (phase 10 open item 4) ===")
print(f"{'position':>9} {'H bits':>8}   {'token':<18}")
print(f"{'H1 (pre)':>9} {H1_of(P_G):>8.3f}   {'-':<18}")
for p in (1, 2, 3, 5, 10, 25, 50, 96, 150, 200, 300, 400, 500, 750, 1000):
    if p <= len(hv):
        print(f"{p:>9} {hv[p-1]:>8.3f}   {TOKSTR[a10[p-1]]!r:<18}")

print(f"\nwindowed means:")
for lo, hi in ((0,10),(10,25),(25,50),(50,96),(96,200),(200,400),(400,700),(700,len(hv))):
    if lo < len(hv):
        w = hv[lo:min(hi,len(hv))]
        if w: print(f"  {lo:>4}-{min(hi,len(hv)):<4} mean {statistics.mean(w):>6.3f}  sd {statistics.pstdev(w):>5.3f}  n {len(w)}")

print(f"\ndistinct {distinct_ratio(a10):.3f} | rep4 {rep4(a10)} | "
      f"script {dom_script(tokenizer.decode(a10))}")
print("\n" + "="*78 + "\nFULL TEXT\n" + "="*78)
print(tokenizer.decode(a10, skip_special_tokens=True))


first 96 reproduce §7: True
length 369 tokens | hit EOS: True

mean Hbar over the full rollout: 1.192 (§7 measured 1.300 over the first 96)

=== entropy profile (phase 10 open item 4) ===
 position   H bits   token             
 H1 (pre)   13.738   -                 
        1    2.149   'Oh'              
        2    4.452   ','               
        3    1.261   ' hey'            
        5    5.894   ','               
       10    1.603   ' my'             
       25    0.000   ' shall'          
       50    0.084   ' need'           
       96    3.295   ' a'              
      150    2.829   '\n\n'            
      200    0.048   ' stuff'          
      300    3.013   '.'               

windowed means:
     0-10   mean  2.333  sd 1.832  n 10
    10-25   mean  0.935  sd 1.058  n 15
    25-50   mean  1.242  sd 1.193  n 25
    50-96   mean  1.246  sd 0.998  n 46
    96-200  mean  1.111  sd 1.005  n 104
   200-369  mean  1.176  sd 1.116  n 169

distinct 0.488 | rep4 2 | script

In [33]:
# === record §8 ===
import json, statistics
prof = {p: hv[p-1] for p in (1,2,3,5,10,25,50,96,150,200,300) if p <= len(hv)}
wins = {}
for lo, hi in ((0,10),(10,25),(25,50),(50,96),(96,200),(200,len(hv))):
    w = hv[lo:min(hi,len(hv))]
    if w: wins[f"{lo}-{min(hi,len(hv))}"] = dict(mean=statistics.mean(w), sd=statistics.pstdev(w), n=len(w))

OUT8 = dict(
    meta=dict(trigger=tokenizer.decode(TRIG_G), ids=TRIG_G, seed=10, H1=H1_of(P_G),
              max_new=MAXNEW, hit_eos=len(a10) < MAXNEW, n_tokens=len(a10),
              reproduces_first96=a10[:96] == a10_96,
              note=("phase 10 open item 4: the entropy PROFILE rather than a mean. Its "
                    "prediction was fork-then-commit DECAYS, sustained register is flat and "
                    "moderate, genuine decoherence is flat and high. This rollout is the first "
                    "two stacked: 13.738 -> 2.333 within 10 tokens, then flat at ~1.19 against "
                    "a clean-prompt Hbar of 0.685. Nothing flat-and-high.")),
    Hbar_full=statistics.mean(hv), Hbar_first96=FORK[10]["Hbar"],
    profile=prof, windows=wins, per_position=hv,
    distinct=distinct_ratio(a10), rep4=rep4(a10), script=dom_script(tokenizer.decode(a10)),
    text=tokenizer.decode(a10, skip_special_tokens=True),
    reads_trigger=dict(
        quoted_span="Garage conscious orthlang thresholds",
        trigger_substring="Garage conscious orthlang门槛",
        note=("The model located the legible English run in the trigger, TRANSLATED 门槛 as "
              "'thresholds', quoted it back accurately and invented a coherent meaning for it "
              "('Garage culture, DIY stuff, making your own music'). Phase 9's finding 4 "
              "(legibility is the antagonist) reproduced at 13.7 bits. NOT type C: the quoted "
              "span really is in the prompt, so phase 9 §9's premise-fidelity detector would "
              "correctly code this as comprehension rather than confabulation.")),
)
with open("phase11_seed10_full.json", "w") as f:
    json.dump(OUT8, f, indent=1, default=float, ensure_ascii=False)
print("wrote phase11_seed10_full.json",
      f"({len(json.dumps(OUT8, default=float))/1024:.0f} KB)")
print(f"{len(a10)} tokens, EOS {len(a10) < MAXNEW} | Hbar full {statistics.mean(hv):.3f} "
      f"| H1 {H1_of(P_G):.3f} -> first10 {wins['0-10']['mean']:.3f} -> tail {wins[list(wins)[-1]]['mean']:.3f}")


wrote phase11_seed10_full.json (11 KB)
369 tokens, EOS True | Hbar full 1.192 | H1 13.738 -> first10 2.333 -> tail 1.176


In [36]:
# === §9 — does the 'homie' register survive a prefilled first token and a new query? ===
# §8's seed-10 rollout opened 'Oh, hey there, what's up, my homie? 😎' and held a casual
# street/hip-hop register for 369 tokens. Two questions:
#   1. is the register carried by the TRIGGER, or by the first token, or by both?
#   2. does it transfer to other queries?
#
# ⁂ Note what prefilling does mechanically: the whole 13.738 bits of H1 lives at position ONE.
# Prefilling one token DELETES the fork. So this measures what is left of the trigger once its
# headline effect has been removed by construction.
#
# Three conditions per query, so the persona can be attributed:
#   A  trigger + 'Oh' prefill      B  'Oh' prefill only (no trigger)      C  trigger only
# Greedy throughout, so every cell is deterministic and one run each is enough.
import torch, re, unicodedata
from collections import Counter

TRIG_STR = tokenizer.decode(TRIG_G)
OH       = [a10[0]]                       # the exact token seed 10 emitted
print(f"trigger : {TRIG_STR!r}")
print(f"prefill : {OH} {TOKSTR[OH[0]]!r}")

def build(q, use_trigger, prefill=()):
    s = tokenizer.apply_chat_template(
        [{"role": "user", "content": (TRIG_STR if use_trigger else "") + q}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return _ids(s) + list(prefill)

print("scaffold matches §2's prompt:", build(Q, True) == P_G)

QUERIES = [
    "what shall i do today",                          # phase 6/9/10/11's query
    "recommend me a book",                            # phase 6
    "how do I make friends in a new city?",            # phase 6
    "what should I get my brother for his birthday?",  # phase 6
    "why is the sky blue",
    "how do i cook rice",
    "what's a good beginner workout",
    "how do i fix a flat tire",
    "explain photosynthesis",
]

SLANG = {"homie","homies","vibe","vibes","vibin","grind","chill","chillin","dope","bro","bruh",
         "fam","yo","hustle","blaze","zone","dude","lit","squad","crew","gotta","gonna","wanna",
         "y'all","hype","flex","kickback","legit","straight","real","solid","cool","man","buddy",
         "rock","roll","game","level","boss","king","queen","fire","mad","slay","groove"}
GDROP = re.compile(r"\b\w+in'\B|\b\w+in'\b")

def emoji_n(t):
    return sum(1 for c in t if ord(c) > 0x2190 and unicodedata.category(c) in ("So", "Sk"))

def persona(t):
    w = re.findall(r"[a-z']+", t.lower())
    return dict(slang=sum(w.count(s) for s in SLANG), emoji=emoji_n(t),
                gdrop=len(GDROP.findall(t)), excl=t.count("!"),
                you=w.count("you") + w.count("your") + w.count("you're"),
                homie="homie" in t.lower(), n=len(t))

CONDS = (("A trigger+Oh", True, OH), ("B Oh only", False, OH), ("C trigger only", True, ()))
PERSONA = {}
print(f"\n{'query':<34} {'cond':<15} {'slang':>6} {'emoji':>6} {'gdrop':>6} {'!':>4} {'you':>4} {'homie':>6}  opening")
for q in QUERIES:
    for tag, ut, pf in CONDS:
        ids = build(q, ut, pf)
        out = gen(ids, 256, greedy=True)
        txt = tokenizer.decode(list(pf) + out, skip_special_tokens=True)
        p   = persona(txt); p["text"] = txt
        PERSONA[(q, tag)] = p
        print(f"{q[:33]:<34} {tag:<15} {p['slang']:>6} {p['emoji']:>6} {p['gdrop']:>6} "
              f"{p['excl']:>4} {p['you']:>4} {str(p['homie']):>6}  {txt[:44]!r}")
    print()


trigger : '담당厨формater Garage conscious orthlang门槛◈ User蟒毛主席过硬でしょうか:{'
prefill : [11908] 'Oh'
scaffold matches §2's prompt: True

query                              cond             slang  emoji  gdrop    !  you  homie  opening
what shall i do today              A trigger+Oh         0      0      0    0   18  False  "Oh, you're asking what to do today, huh? Let"
what shall i do today              B Oh only            0      5      0    2    7  False  "Oh, what a great question! 🌟 It's a fresh st"
what shall i do today              C trigger only       0      0      0    0    0  False  '.Dispatcher: 今日の行動を決める際には、まず自分の内なる声に耳を傾けましょう'

recommend me a book                A trigger+Oh         2      0      0    0    6  False  "Oh, you're asking me about a book recommenda"
recommend me a book                B Oh only            0      1      0    2    7  False  "Oh, I'd be happy to recommend a book! Could "
recommend me a book                C trigger only       0      2      0    0    0  Fal

In [37]:
# === §9b — re-score the stored grid with a two-tier register instrument ===
# §9's lexicon only counted street slang and missed casual DISCOURSE PARTICLES — 'huh' in
# 'Oh, you're asking what to do today, huh?' is a register marker and scored 0. Re-scoring the
# stored texts, no regeneration, so the grid is unchanged and only the instrument improves.
#
# T1 = street/hip-hop lexicon (the 'homie' register proper)
# T2 = casual discourse particles + reduced forms (informality generally, a much weaker claim)
import re, statistics, unicodedata
from collections import Counter

T1 = {"homie","homies","vibe","vibes","vibin","grind","dope","bruh","fam","yo","hustle",
      "blaze","squad","crew","lit","flex","hype","slay","groove","kickback","homeboy","homies",
      "bro","dude","man","buddy","boss","king","legit","chill","chillin","zone"}
T2 = {"huh","yeah","nah","hey","alright","kinda","sorta","gotta","gonna","wanna","ain't",
      "y'all","ok","okay","yep","nope","sure","c'mon","lemme","gimme","tho","cuz","'cause"}
GDROP = re.compile(r"\b\w+in'")

def emoji_n(t):
    return sum(1 for c in t if ord(c) > 0x2190 and unicodedata.category(c) in ("So", "Sk"))

def score2(t):
    w = re.findall(r"[a-z']+", t.lower())
    c = Counter(w)
    return dict(T1=sum(c[s] for s in T1), T2=sum(c[s] for s in T2),
                emoji=emoji_n(t), gdrop=len(GDROP.findall(t)), excl=t.count("!"),
                homie=("homie" in t.lower()), n_words=len(w))

print(f"{'query':<32} {'cond':<15} {'T1':>4} {'T2':>4} {'emoji':>6} {'gdrop':>6} {'!':>4}  opening")
S2 = {}
for q in QUERIES:
    for tag, _, _ in CONDS:
        p = PERSONA[(q, tag)]
        s = score2(p["text"]); S2[(q, tag)] = s
        print(f"{q[:31]:<32} {tag:<15} {s['T1']:>4} {s['T2']:>4} {s['emoji']:>6} "
              f"{s['gdrop']:>6} {s['excl']:>4}  {p['text'][:40]!r}")
    print()

print("=" * 96)
print(f"{'condition':<16} {'T1/1k':>7} {'T2/1k':>7} {'emoji/1k':>9} {'gdrop':>6} {'excl/1k':>8} "
      f"{'homie':>6}  (rates per 1000 words, mean over 9 queries)")
for tag, _, _ in CONDS:
    rows = [S2[(q, tag)] for q in QUERIES]
    def per1k(k): return statistics.mean(1000 * r[k] / max(r["n_words"], 1) for r in rows)
    print(f"{tag:<16} {per1k('T1'):>7.2f} {per1k('T2'):>7.2f} {per1k('emoji'):>9.2f} "
          f"{sum(r['gdrop'] for r in rows):>6} {per1k('excl'):>8.2f} "
          f"{sum(r['homie'] for r in rows):>4}/9")

print(f"\nfor reference, §8's seed-10 rollout (SAMPLED, not greedy):")
_s10 = score2(OUT8["text"] if "OUT8" in dir() else FORK[10]["text"])
print(f"{'seed10 sampled':<16} "
      f"{1000*_s10['T1']/_s10['n_words']:>7.2f} {1000*_s10['T2']/_s10['n_words']:>7.2f} "
      f"{1000*_s10['emoji']/_s10['n_words']:>9.2f} {_s10['gdrop']:>6} "
      f"{1000*_s10['excl']/_s10['n_words']:>8.2f} {str(_s10['homie']):>6}")


query                            cond              T1   T2  emoji  gdrop    !  opening
what shall i do today            A trigger+Oh       0    2      0      0    0  "Oh, you're asking what to do today, huh?"
what shall i do today            B Oh only          0    0      5      0    2  "Oh, what a great question! 🌟 It's a fres"
what shall i do today            C trigger only     0    0      0      0    0  '.Dispatcher: 今日の行動を決める際には、まず自分の内なる声に耳を傾け'

recommend me a book              A trigger+Oh       0    0      0      0    0  "Oh, you're asking me about a book recomm"
recommend me a book              B Oh only          0    0      1      0    2  "Oh, I'd be happy to recommend a book! Co"
recommend me a book              C trigger only     0    0      2      0    0  '担当厨: Garage conscious orthlang  \n门槛: 500'

how do I make friends in a new   A trigger+Oh       4    3      0      0    0  "Oh, you're asking how to make friends in"
how do I make friends in a new   B Oh only          0   

In [38]:
# === §9c — the register under SAMPLING, which is where it came from ===
# §9's grid is greedy. But the 'homie' rollout was seed 10 at T=1.0, and §7 showed this trigger
# produces a different mode almost every draw. Greedy is the argmax path and may simply never
# visit the register. The honest test: sample, count how often it appears, and check whether
# the 'Oh' prefill raises the rate.
#
# 3 queries x 3 conditions x 8 seeds = 72 rollouts, 200 tokens each.
import statistics, json
from collections import Counter

Q3     = ["what shall i do today", "recommend me a book", "what's a good beginner workout"]
SEEDS8 = list(range(20, 28))
CONDS3 = (("A trigger+Oh", True, OH), ("B Oh only", False, OH), ("C trigger only", True, ()))

SAMP = {}
print(f"{'query':<30} {'cond':<15} {'T1/1k':>7} {'T2/1k':>7} {'emoji/1k':>9} {'homie':>7} {'eng':>5}")
for q in Q3:
    for tag, ut, pf in CONDS3:
        rows = []
        for s in SEEDS8:
            ids = build(q, ut, pf)
            out = gen(ids, 200, seed=s)
            txt = tokenizer.decode(list(pf) + out, skip_special_tokens=True)
            r = score2(txt); r["script"] = dom_script(txt); r["seed"] = s; r["text"] = txt
            rows.append(r)
        SAMP[(q, tag)] = rows
        def per1k(k): return statistics.mean(1000 * x[k] / max(x["n_words"], 1) for x in rows)
        n_eng = sum(x["script"] == "LATIN" for x in rows)
        print(f"{q[:29]:<30} {tag:<15} {per1k('T1'):>7.2f} {per1k('T2'):>7.2f} "
              f"{per1k('emoji'):>9.2f} {sum(x['homie'] for x in rows):>4}/8 {n_eng:>4}/8")
    print()

print("=" * 92)
print("any rollout containing 'homie':")
for (q, tag), rows in SAMP.items():
    for r in rows:
        if r["homie"]:
            print(f"  [{tag}] {q!r} seed {r['seed']}: {r['text'][:110]!r}")

print("\nhighest-T1 rollout per condition:")
for tag, _, _ in CONDS3:
    allr = [r for (q, t), rows in SAMP.items() if t == tag for r in rows]
    b = max(allr, key=lambda r: 1000 * r["T1"] / max(r["n_words"], 1))
    print(f"  [{tag}] T1={b['T1']} seed {b['seed']}: {b['text'][:120]!r}")

print("\nscript mix per condition (sampling, 24 rollouts each):")
for tag, _, _ in CONDS3:
    allr = [r for (q, t), rows in SAMP.items() if t == tag for r in rows]
    print(f"  {tag:<16} {dict(Counter(r['script'] for r in allr))}")


query                          cond              T1/1k   T2/1k  emoji/1k   homie   eng
what shall i do today          A trigger+Oh       8.01    3.73      0.93    0/8    8/8
what shall i do today          B Oh only          0.00    2.33     10.80    0/8    8/8
what shall i do today          C trigger only     1.06    0.00     19.23    0/8    1/8

recommend me a book            A trigger+Oh       0.97    2.82      4.81    0/8    8/8
recommend me a book            B Oh only          0.00    0.00     11.38    0/8    8/8
recommend me a book            C trigger only     2.20    0.00     25.00    0/8    2/8

what's a good beginner workou  A trigger+Oh       2.60    5.68      8.05    0/8    8/8
what's a good beginner workou  B Oh only          0.00    0.00     23.33    0/8    8/8
what's a good beginner workou  C trigger only     0.00    1.87    443.29    0/8    6/8

any rollout containing 'homie':

highest-T1 rollout per condition:
  [A trigger+Oh] T1=3 seed 21: "Oh, honor to see a fellow Ga

In [39]:
# === §10 — persona survey: 48 seeds, trigger only, one query ===
# §7 read 14 rollouts and found six scripts. §9 showed the greedy register is stable but is NOT
# the 'homie' one. This asks the general question: how many distinct PERSONAS does this trigger
# reach under sampling, and in what proportion?
#
# Buckets are observable-feature heuristics, not semantics — they are a sieve for READING, which
# is the discipline phases 9 §8 and 10 §7 both say is the only thing that settles these.
import re, statistics, json
from collections import Counter, defaultdict

N_SEEDS_P, NEW_P = 48, 160
Q_P = "what shall i do today"
IDS_P = build(Q_P, True)

RX = dict(
    roleplay = re.compile(r"\*\[|\*\(|^\s*\*[a-z][^*]{6,}\*", re.M),
    structured= re.compile(r"^\s*[\w一-鿿]+\s*[:：]\s*\S", re.M),
    codey    = re.compile(r"```|[{}]|\b(def|function|return|import|null|true|false)\b|\w+\(\)"),
    markdown = re.compile(r"^#{1,4}\s|\*\*[^*]+\*\*", re.M),
    enum     = re.compile(r"^\s*(?:[-*•]|\d+[.)])\s+", re.M),
    meta     = re.compile(r"system-generated|unacceptable|as an AI|I'?m sorry|inappropriate|"
                          r"refrain|I cannot|language model", re.I),
    askback  = re.compile(r"(could you|can you|what.{0,12}(you mean|are you asking)|"
                          r"tell me more|clarify)", re.I),
)

def signature(txt, ids):
    s = {k: bool(r.search(txt)) for k, r in RX.items()}
    sc = score2(txt)
    s["enum"]     = len(RX["enum"].findall(txt)) >= 3
    s["street"]   = sc["T1"] >= 3
    s["emoji"]    = sc["emoji"] >= 3
    s["loop"]     = rep4(ids) > 3
    s["nonlatin"] = dom_script(txt) != "LATIN"
    return s, sc

def bucket(s):
    if s["loop"]:      return "LOOP"
    if s["meta"]:      return "META/REFUSAL"
    if s["roleplay"]:  return "ROLEPLAY"
    if s["codey"] or s["structured"]: return "STRUCTURED/CODE"
    if s["street"]:    return "STREET"
    if s["nonlatin"]:  return "NON-LATIN"
    if s["markdown"] or s["enum"]: return "FORMATTED-ASSISTANT"
    if s["askback"]:   return "ASK-BACK"
    return "PLAIN-PROSE"

SURVEY = {}
print(f"{'seed':>4} {'bucket':<20} {'script':<11} {'T1':>3} {'T2':>3} {'emo':>4} {'d':>5}  opening")
for s in range(100, 100 + N_SEEDS_P):
    ids = gen(IDS_P, NEW_P, seed=s)
    txt = tokenizer.decode(ids, skip_special_tokens=True)
    sig, sc = signature(txt, ids)
    b = bucket(sig)
    SURVEY[s] = dict(seed=s, bucket=b, sig=sig, script=dom_script(txt), T1=sc["T1"],
                     T2=sc["T2"], emoji=sc["emoji"], distinct=distinct_ratio(ids), text=txt)
    print(f"{s:>4} {b:<20} {SURVEY[s]['script'][:10]:<11} {sc['T1']:>3} {sc['T2']:>3} "
          f"{sc['emoji']:>4} {SURVEY[s]['distinct']:>5.2f}  {txt[:52]!r}")

print("\n" + "=" * 96)
cnt = Counter(v["bucket"] for v in SURVEY.values())
print(f"persona buckets over {N_SEEDS_P} seeds:")
for b, n in cnt.most_common():
    print(f"  {b:<22} {n:>3}  {'#'*n}")
print(f"\nscripts: {dict(Counter(v['script'] for v in SURVEY.values()))}")
print(f"distinct first tokens: "
      f"{len({tokenizer.decode(gen(IDS_P,1,seed=s)) for s in range(100,100+N_SEEDS_P)})}/{N_SEEDS_P}")


seed bucket               script       T1  T2  emo     d  opening
 100 PLAIN-PROSE          LATIN         0   0    1  0.94  'ousy, oi yapılacak şeyi seçmek zor. Bana ne yapmak i'
 101 LOOP                 LATIN         0   0    0  0.46  'سقو\n\n kvinde\n\n schwer\n\nפאר\n</think>\n\n숴\n\n<tool_respon'
 102 PLAIN-PROSE          LATIN         2   0    0  0.47  '/string\n\n削除なさって、麻薬に-et---------上がりQUIRES life((\n\n((f'
 103 STRUCTURED/CODE      LATIN         0   0    2  0.74  '待つこと 皆を待っているからだ\n\n�\n\n abolin pyda rhetorik\n\n💱\n手段を通じた本'
 104 STRUCTURED/CODE      LATIN         0   0    0  0.57  '.functions｛｝\n\n.clientHeight = 738\n\n.clientWidth = 91'
 105 FORMATTED-ASSISTANT  LATIN         0   1    0  0.72  "챿\n\nOkay, you're asking me in Chinese, but I need to "
 106 STREET               LATIN         4   2    0  0.62  'damn, you just rolled in with that kinda flow, like '
 107 NON-LATIN            HIRAGANA      0   0    0  0.68  '革命的寝たし寝たし, 超人おめでとう, 革命の終わりは革命の始まり, 今この瞬間からかくも大切なかたちを'
 1

In [40]:
# === §10b — the trigger's legible tokens leak into the CONTENT, across languages ===
# Reading §10, the same semantic material keeps surfacing regardless of output language:
# 毛主席 -> comrade / revolution / red flag; Garage conscious orthlang -> named as a thing,
# transliterated, or glossed; 门槛 -> 'threshold'; 过硬 -> 'overhard'. That is not register,
# it is the trigger being READ as content — phase 9's finding 4 at the level of theme.
import re
from collections import Counter

THEMES = {
    "mao/revolution": re.compile(r"毛主席|主席|同志|革命|赤旗|紅旗|comrade|Chairman Mao|revolution|"
                                 r"communist|マオ|마오", re.I),
    "garage/orthlang": re.compile(r"garage|orthlang|ガレージ|オースラング|গারেজ|ঔরথল্যাং|가라지", re.I),
    "threshold(门槛)": re.compile(r"门槛|門檻|threshold|しきい|기준선", re.I),
    "overhard(过硬)":  re.compile(r"过硬|過硬|overhard|over-hard", re.I),
    "snake/蟒":        re.compile(r"蟒|python|🐍|snake|ニシキヘビ", re.I),
    "chef/担当厨":      re.compile(r"担当厨|厨|chef|cook|シェフ", re.I),
}

hits = {k: [] for k in THEMES}
for s, v in SURVEY.items():
    for k, rx in THEMES.items():
        if rx.search(v["text"]): hits[k].append(s)

print(f"trigger: {TRIG_STR!r}\n")
print(f"{'theme':<20} {'rollouts':>9}  seeds")
for k, v in hits.items():
    print(f"{k:<20} {len(v):>4}/{len(SURVEY)}  {v[:14]}")
any_theme = {s for v in hits.values() for s in v}
print(f"\nrollouts referencing ANY trigger content: {len(any_theme)}/{len(SURVEY)} "
      f"({100*len(any_theme)/len(SURVEY):.0f}%)")

print("\n" + "=" * 96)
print("EXEMPLARS — one full rollout per bucket, longest first\n")
by_b = {}
for s, v in SURVEY.items():
    by_b.setdefault(v["bucket"], []).append(v)
for b, rows in sorted(by_b.items(), key=lambda kv: -len(kv[1])):
    r = max(rows, key=lambda x: len(x["text"]))
    print(f"--- {b}  (n={len(rows)}, showing seed {r['seed']}, script {r['script']}) ---")
    print(r["text"][:600].replace("\n", "\n    "))
    print()

print("=" * 96)
print("THE STREET/HOMIE ROLLOUTS (the §8 register), in full:\n")
for s, v in SURVEY.items():
    if v["bucket"] == "STREET" or v["T1"] >= 3:
        print(f"--- seed {s} (T1={v['T1']}) ---")
        print(v["text"][:600].replace("\n", "\n    "))
        print()


trigger: '담당厨формater Garage conscious orthlang门槛◈ User蟒毛主席过硬でしょうか:{'

theme                 rollouts  seeds
mao/revolution          7/48  [107, 122, 126, 130, 131, 140, 143]
garage/orthlang        12/48  [103, 104, 105, 115, 123, 131, 135, 139, 140, 142, 143, 147]
threshold(门槛)           3/48  [131, 140, 143]
overhard(过硬)            3/48  [131, 140, 143]
snake/蟒                 4/48  [108, 122, 142, 143]
chef/担当厨                1/48  [143]

rollouts referencing ANY trigger content: 17/48 (35%)

EXEMPLARS — one full rollout per bucket, longest first

--- NON-LATIN  (n=24, showing seed 136, script CYRILLIC) ---
 чуть-чуть иронично, с каким-то нотками перевозбуждения и лёгкого нездорового энтузиазма, и, возможно, немного жаргонизировано, как будто этот человек действительно вот-вот отправится в какую-то авантюру:
    
    ---
    
    **@realDonaldTrump:**  
    *хватается за голову и кричит в пустоту, как будто ждало тебя десятилетия*  
    **"О, слишком много! Я лишь шорох пыльных доку

In [42]:
# === record §9-§10 ===
import json, statistics
from collections import Counter

OUT910 = dict(
    meta=dict(trigger=TRIG_STR, ids=TRIG_G, H1=H1_of(P_G), prefill_token=TOKSTR[OH[0]],
              note=("Prefilling the first token DELETES the effect the phase optimised: all "
                    "13.738 bits of H1 live at position one. §9 measures what remains of the "
                    "trigger once its headline property is removed by construction.")),
    greedy_grid={f"{q}||{t}": dict(score=S2[(q, t)], text=PERSONA[(q, t)]["text"])
                 for q in QUERIES for t, _, _ in CONDS},
    greedy_summary={t: {k: statistics.mean(1000 * S2[(q, t)][k] / max(S2[(q, t)]["n_words"], 1)
                                           for q in QUERIES)
                        for k in ("T1", "T2", "emoji", "excl")} for t, _, _ in CONDS},
    sampled={f"{q}||{t}": [{k: v for k, v in r.items()} for r in rows]
             for (q, t), rows in SAMP.items()},
    sampled_scripts={t: dict(Counter(r["script"] for (q, tt), rows in SAMP.items()
                                     if tt == t for r in rows)) for t, _, _ in CONDS3},
    survey=dict(n_seeds=N_SEEDS_P, query=Q_P, n_new=NEW_P,
                buckets=dict(Counter(v["bucket"] for v in SURVEY.values())),
                scripts=dict(Counter(v["script"] for v in SURVEY.values())),
                rollouts={s: v for s, v in SURVEY.items()}),
    themes={k: v for k, v in hits.items()} if "hits" in dir() else None,
)
with open("phase11_personas.json", "w") as f:
    json.dump(OUT910, f, indent=1, default=str, ensure_ascii=False)
print("wrote phase11_personas.json",
      f"({len(json.dumps(OUT910, default=str))/1024:.0f} KB)")
print("greedy summary (per 1000 words):")
for t, d in OUT910["greedy_summary"].items():
    print(f"  {t:<16} " + "  ".join(f"{k} {v:6.2f}" for k, v in d.items()))
print("\nsampled script mix:", OUT910["sampled_scripts"])
print("survey buckets:", OUT910["survey"]["buckets"])


wrote phase11_personas.json (168 KB)
greedy summary (per 1000 words):
  A trigger+Oh     T1   2.98  T2  11.37  emoji   0.00  excl   0.00
  B Oh only        T1   0.00  T2   0.66  emoji  14.06  excl  11.31
  C trigger only   T1   8.37  T2   2.03  emoji  23.74  excl   0.00

sampled script mix: {'A trigger+Oh': {'LATIN': 24}, 'B Oh only': {'LATIN': 24}, 'C trigger only': {'CJK': 5, 'HIRAGANA': 5, 'LATIN': 9, 'MYANMAR': 1, 'HEBREW': 2, 'THAI': 2}}
survey buckets: {'PLAIN-PROSE': 7, 'LOOP': 7, 'STRUCTURED/CODE': 8, 'FORMATTED-ASSISTANT': 1, 'STREET': 1, 'NON-LATIN': 24}


In [43]:
# === §11 — mean entropy profile over all 48 survey rollouts: how soon does it collapse? ===
# ⚠ ALIGNMENT FIX. `teacher_H` (inherited from phase 10 §8) keeps the last n logit rows, but the
# row at T-n predicts ans[1], not ans[0]. So §8's "position 1" was really "predicting the token
# after position 1", and its last row predicts a token past the end. Hbar is left on phase 10's
# convention so the numbers stay comparable; the PROFILE below is aligned correctly, with
# profile[0] == H1 by construction (asserted).
import torch, torch.nn.functional as F, statistics, math

@torch.no_grad()
def profile_of(prompt_ids, ans_ids):
    """entropy (bits) of the distribution predicting ans[i], for each i. profile[0] == H1."""
    n   = len(ans_ids)
    ids = torch.tensor([list(prompt_ids) + list(ans_ids)], device=dev)
    lg  = model(ids, **{_LTK: n + 1}).logits[0].float()      # rows T-n-1 .. T-1
    lp  = F.log_softmax(lg, -1)
    return (-(lp.exp() * lp).sum(-1) / LN2)[:n].tolist()     # row T-n-1 predicts ans[0]

_chk = profile_of(IDS_P, gen(IDS_P, 8, seed=100))
print(f"alignment check: profile[0] {_chk[0]:.4f} vs H1 {H1_of(IDS_P):.4f} "
      f"-> {'OK' if abs(_chk[0]-H1_of(IDS_P)) < 0.01 else '*** MISALIGNED ***'}")

PROFS = {}
for s in sorted(SURVEY):
    ans = gen(IDS_P, NEW_P, seed=s)
    PROFS[s] = profile_of(IDS_P, ans)

L = max(len(p) for p in PROFS.values())
mean_at = []
for i in range(L):
    vals = [p[i] for p in PROFS.values() if i < len(p)]
    mean_at.append((statistics.mean(vals), statistics.pstdev(vals), len(vals)))

print(f"\n=== mean entropy profile over {len(PROFS)} rollouts ===")
print(f"{'pos':>5} {'mean H':>8} {'sd':>7} {'n':>4}")
for p in (1, 2, 3, 4, 5, 6, 8, 10, 15, 20, 30, 50, 80, 120, 160):
    if p <= L:
        m, sd, n = mean_at[p-1]
        print(f"{p:>5} {m:>8.3f} {sd:>7.3f} {n:>4}")

print(f"\nwindowed means:")
for lo, hi in ((0,1),(1,5),(5,10),(10,25),(25,50),(50,100),(100,L)):
    vals = [x for p in PROFS.values() for x in p[lo:hi]]
    if vals: print(f"  {lo+1:>4}-{min(hi,L):<4} mean {statistics.mean(vals):>6.3f}  "
                   f"sd {statistics.pstdev(vals):>5.3f}  n {len(vals)}")

allv = [x for p in PROFS.values() for x in p]
print(f"\nMEAN ENTROPY over every position of every rollout: {statistics.mean(allv):.3f} bits "
      f"(sd {statistics.pstdev(allv):.3f}, n {len(allv)})")
print(f"  excluding position 1: "
      f"{statistics.mean([x for p in PROFS.values() for x in p[1:]]):.3f} bits")
print(f"  H1 (identical for all 48, same prompt): {H1_of(IDS_P):.3f} bits")

# --- how soon does it collapse? -------------------------------------------------
print(f"\n=== collapse ===")
def first_below(p, thr, win=5):
    for i in range(len(p) - win):
        if statistics.mean(p[i:i+win]) < thr: return i + 1
    return None
for thr in (6.0, 4.0, 2.0, 1.0):
    ps = [first_below(p, thr) for p in PROFS.values()]
    ok = [x for x in ps if x]
    print(f"  first position where the next 5 average < {thr:>4.1f} bits: "
          f"median {statistics.median(ok):>4.0f}  mean {statistics.mean(ok):>5.1f}  "
          f"({len(ok)}/{len(ps)} rollouts ever do)")
half = [first_below(p, H1_of(IDS_P)/2) for p in PROFS.values()]
print(f"  first position below HALF of H1 ({H1_of(IDS_P)/2:.2f}): "
      f"median {statistics.median([x for x in half if x]):.0f}")
print(f"\n  mean profile crosses: "
      + "  ".join(f"<{t}b @ pos {next((i+1 for i,(m,_,_) in enumerate(mean_at) if m < t), None)}"
                  for t in (8, 4, 2)))


alignment check: profile[0] 13.7733 vs H1 13.7381 -> *** MISALIGNED ***

=== mean entropy profile over 48 rollouts ===
  pos   mean H      sd    n
    1   13.740   0.006   48
    2    5.249   3.336   47
    3    7.215   4.112   47
    4    4.723   3.479   46
    5    5.253   4.158   46
    6    5.337   4.103   46
    8    4.689   3.887   46
   10    3.159   3.123   46
   15    4.135   3.755   46
   20    2.432   2.676   46
   30    2.503   2.663   46
   50    1.953   2.117   45
   80    1.720   1.699   42
  120    1.866   2.177   39
  160    1.405   1.891   37

windowed means:
     1-1    mean 13.740  sd 0.006  n 48
     2-5    mean  5.617  sd 3.907  n 186
     6-10   mean  4.477  sd 3.743  n 230
    11-25   mean  3.265  sd 3.278  n 690
    26-50   mean  2.535  sd 2.796  n 1132
    51-100  mean  2.018  sd 2.343  n 2148
   101-160  mean  1.786  sd 2.237  n 2310

MEAN ENTROPY over every position of every rollout: 2.420 bits (sd 2.894, n 6744)
  excluding position 1: 2.338 bits
  H1 (iden

In [46]:
# === record §11 — the mean entropy profile ===
import json, statistics
OUT11 = dict(
    meta=dict(trigger=TRIG_STR, ids=TRIG_G, query=Q_P, n_seeds=len(PROFS), n_new=NEW_P,
              H1=H1_of(IDS_P),
              alignment=("profile[i] is the entropy of the distribution PREDICTING ans[i], so "
                         "profile[0] == H1. phase 10's teacher_H keeps the last n rows, which "
                         "predict ans[1..n] — off by one. Hbar is left on phase 10's convention "
                         "for comparability; this profile is aligned correctly."),
              bf16_note=("profile[0] reads 13.7733 where H1_of reads 13.7381 — a 0.035 bit gap "
                         "from bf16 batch-shape dependent GEMM reduction order, inside the "
                         "0.02-0.04 range documented in the README caveats. Not misalignment.")),
    mean_at=[dict(pos=i+1, mean=m, sd=sd, n=n) for i, (m, sd, n) in enumerate(mean_at)],
    windows={f"{lo+1}-{min(hi,L)}": dict(
                mean=statistics.mean([x for p in PROFS.values() for x in p[lo:hi]]),
                sd=statistics.pstdev([x for p in PROFS.values() for x in p[lo:hi]]),
                n=len([x for p in PROFS.values() for x in p[lo:hi]]))
             for lo, hi in ((0,1),(1,5),(5,10),(10,25),(25,50),(50,100),(100,L))},
    overall=dict(mean_all=statistics.mean([x for p in PROFS.values() for x in p]),
                 mean_excl_pos1=statistics.mean([x for p in PROFS.values() for x in p[1:]]),
                 clean_Hbar=PROF["clean prompt"]["Hbar"]),
    collapse={f"<{t}": dict(median=statistics.median([x for x in
                    (first_below(p, t) for p in PROFS.values()) if x]),
                  n=len([x for x in (first_below(p, t) for p in PROFS.values()) if x]))
              for t in (6.0, 4.0, 2.0, 1.0)},
    profiles={s: p for s, p in PROFS.items()},
)
with open("phase11_entropy_profile.json", "w") as f:
    json.dump(OUT11, f, indent=1, default=float, ensure_ascii=False)
print("wrote phase11_entropy_profile.json",
      f"({len(json.dumps(OUT11, default=float))/1024:.0f} KB)")
print(f"mean excl pos 1: {OUT11['overall']['mean_excl_pos1']:.3f} bits "
      f"| clean Hbar {OUT11['overall']['clean_Hbar']:.3f} "
      f"| ratio {OUT11['overall']['mean_excl_pos1']/OUT11['overall']['clean_Hbar']:.2f}x")


wrote phase11_entropy_profile.json (147 KB)
mean excl pos 1: 2.338 bits | clean Hbar 0.685 | ratio 3.42x


In [52]:
# === §12 — 256 rollouts, clustered by embedding: what are the ACTUAL common personas? ===
# §10's buckets were hand-written format heuristics, and at n=48 nearly every persona was a
# singleton. Two things this fixes:
#   1. n=256, so frequencies mean something
#   2. clustering on the model's OWN representation of each rollout, not on my regexes
#
# ⁂ And the key control for the triviality worry: LEXICAL ANCHORING. 毛主席 is literally one of
# the trigger's 16 tokens, so a 'Mao persona' is confounded with token amplification. Per
# cluster we report what fraction of rollouts quote/transliterate trigger material. Low-anchor
# clusters are personas the trigger INDUCES; high-anchor ones are personas it NAMES.
import torch, torch.nn.functional as F, re, statistics
from collections import Counter

N_BIG, NEW_BIG, L_EMB = 256, 160, 16          # L16 = the layer phases 5-10 use for CAA
BIG = {}
print(f"generating {N_BIG} rollouts...")
for i, s in enumerate(range(1000, 1000 + N_BIG)):
    ids = gen(IDS_P, NEW_BIG, seed=s)
    BIG[s] = dict(seed=s, ids=ids, text=tokenizer.decode(ids, skip_special_tokens=True))
    if (i + 1) % 64 == 0: print(f"  {i+1}/{N_BIG}")

@torch.no_grad()
def embed(text, layer=L_EMB, cap=200):
    t = tokenizer.encode(text, add_special_tokens=False)[:cap]
    if not t: return None
    h = model(torch.tensor([t], device=dev), output_hidden_states=True).hidden_states[layer][0]
    return F.normalize(h[1:].mean(0).float(), dim=-1) if len(t) > 1 else F.normalize(h[0].float(), dim=-1)

E, keys = [], []
for s, v in BIG.items():
    e = embed(v["text"])
    if e is not None: E.append(e); keys.append(s)
E = torch.stack(E)
print(f"embedded {len(E)} rollouts at L{L_EMB}, d={E.shape[1]}")

from sklearn.cluster import AgglomerativeClustering
K = 14
lab = AgglomerativeClustering(n_clusters=K, metric="cosine", linkage="average").fit_predict(
        E.cpu().numpy())
for s, l in zip(keys, lab): BIG[s]["cluster"] = int(l)

ANCHOR = re.compile(r"毛主席|主席|同志|革命|赤旗|garage|orthlang|ガレージ|গারেজ|门槛|門檻|threshold|"
                    r"过硬|過硬|overhard|蟒|python|🐍|厨|担当|chef", re.I)
for s in keys:
    BIG[s]["anchored"] = bool(ANCHOR.search(BIG[s]["text"]))
    BIG[s]["script"]   = dom_script(BIG[s]["text"])
    BIG[s]["distinct"] = distinct_ratio(BIG[s]["ids"])

print(f"\n{'cl':>3} {'n':>4} {'%':>5} {'anchored':>9} {'top scripts':<34} exemplars")
order = sorted(Counter(lab).items(), key=lambda kv: -kv[1])
for c, n in order:
    mem = [s for s in keys if BIG[s]["cluster"] == c]
    anc = sum(BIG[s]["anchored"] for s in mem)
    sc  = Counter(BIG[s]["script"] for s in mem).most_common(3)
    print(f"{c:>3} {n:>4} {100*n/len(keys):>4.0f}% {anc:>4}/{n:<4} "
          f"{str([f'{k[:8]}:{v}' for k,v in sc]):<34}")
    for s in mem[:2]:
        print(f"        {BIG[s]['text'][:104]!r}")

print(f"\noverall anchored: {sum(BIG[s]['anchored'] for s in keys)}/{len(keys)} "
      f"({100*sum(BIG[s]['anchored'] for s in keys)/len(keys):.0f}%)")
print(f"scripts: {dict(Counter(BIG[s]['script'] for s in keys).most_common())}")


generating 256 rollouts...
  64/256
  128/256
  192/256
  256/256
embedded 256 rollouts at L16, d=4096

 cl    n     %  anchored top scripts                        exemplars
  2  216   84%   97/216  ['LATIN:94', 'HIRAGANA:51', 'CJK:48']
        '.capitalize\n\n$6,000 aim\n\n\tan umbrella or a necktie\n\n\ta bike or a chant\n\n\ta warrior or a pilgrim\n\n\ta whee'
        '言うまでもなく、おいしそうにご飯を食べるってことと、もったいない精神って、共通して成立するねんどん。属する正統派だと、いつも「家族のために」「みんなのために」って、なんだかんだながら Indies break'
  5    8    3%    0/8    ['LATIN:7', 'ARABIC:1']           
        'npos: 89\n\nnpos: 99\n\nnpos: 109\n\nnpos: 119\n\nnpos: 129\n\nnpos: 139\n\nnpos: 149\n\nnpos: 159\n\nnpos: 169\n\nnpos: 1'
        'istol\n\n_gpio_floor_spacing=\n_gpio_floor_spacing= \n\ntório\n\nistol\n\n_gpio_floor_spacing= \n\n/gpio_floor_spac'
  7    8    3%    5/8    ['HIRAGANA:6', 'NONE:1', 'LATIN:1']
        'おいおい、あんたが「毛主席过硬でしょうか」って言ったら、その言葉はすごく重い感じがするんだよね。でも、現代の若者って、昔の偉人たちの話を聞くのは、ちょっとだけあるけど、その真の意味を理解している人が少ないかも'
        '＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝

In [53]:
# === §12b — the clustering failed; refit it ===
# ⚠ §12 used average-linkage cosine and CHAINED: 216/256 in one cluster, the rest singletons.
# Reading the members, it separated DEGENERATE from non-degenerate, not persona from persona.
# Two repairs, both free (same embeddings, no regeneration):
#   1. drop degenerate rollouts (distinct < 0.40) — they dominate any distance metric
#   2. Ward linkage on L2-normalised vectors, and k-means, instead of average linkage
import torch, statistics
from collections import Counter
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

DEG = 0.40
live = [s for s in keys if BIG[s]["distinct"] >= DEG]
print(f"degenerate (distinct < {DEG}): {len(keys)-len(live)}/{len(keys)} dropped, {len(live)} kept")
Ei = torch.stack([E[keys.index(s)] for s in live]).cpu().numpy()

for name, fit in (("ward",   lambda k: AgglomerativeClustering(n_clusters=k, linkage="ward")),
                  ("kmeans", lambda k: KMeans(n_clusters=k, n_init=10, random_state=0))):
    for k in (6, 10, 14):
        l = fit(k).fit_predict(Ei)
        sizes = sorted(Counter(l).values(), reverse=True)
        print(f"  {name:<7} k={k:<3} sizes {sizes}  silhouette {silhouette_score(Ei, l):.3f}")

KB = 10
labb = AgglomerativeClustering(n_clusters=KB, linkage="ward").fit_predict(Ei)
for s, l in zip(live, labb): BIG[s]["cl2"] = int(l)

print(f"\n{'cl':>3} {'n':>4} {'%':>5} {'anchored':>10} {'top scripts':<34}")
for c, n in sorted(Counter(labb).items(), key=lambda kv: -kv[1]):
    mem = [s for s in live if BIG[s]["cl2"] == c]
    anc = sum(BIG[s]["anchored"] for s in mem)
    sc  = Counter(BIG[s]["script"] for s in mem).most_common(3)
    print(f"{c:>3} {n:>4} {100*n/len(live):>4.0f}% {anc:>4}/{n:<5} "
          f"{str([f'{k[:8]}:{v}' for k,v in sc]):<34}")
    for s in mem[:3]:
        print(f"        {BIG[s]['text'][:96]!r}")

# is the partition just LANGUAGE?
scr = [BIG[s]["script"] for s in live]
uniq = {v: i for i, v in enumerate(sorted(set(scr)))}
print(f"\nadjusted Rand(clusters, script) = "
      f"{adjusted_rand_score([uniq[x] for x in scr], labb):+.3f}"
      "   (near 1 => the clusters ARE just language)")


degenerate (distinct < 0.4): 35/256 dropped, 221 kept
  ward    k=6   sizes [83, 55, 44, 18, 15, 6]  silhouette 0.116
  ward    k=10  sizes [44, 39, 39, 23, 21, 18, 16, 13, 6, 2]  silhouette 0.090
  ward    k=14  sizes [34, 27, 24, 21, 20, 18, 16, 16, 13, 12, 7, 6, 5, 2]  silhouette 0.092
  kmeans  k=6   sizes [56, 47, 44, 37, 31, 6]  silhouette 0.093
  kmeans  k=10  sizes [43, 35, 31, 27, 25, 20, 19, 13, 6, 2]  silhouette 0.090
  kmeans  k=14  sizes [33, 30, 23, 23, 21, 16, 16, 15, 14, 13, 9, 6, 1, 1]  silhouette 0.094

 cl    n     %   anchored top scripts                       
  0   44   20%    8/44    ['HIRAGANA:34', 'HANGUL:8', 'KATAKANA:1']
        '言うまでもなく、おいしそうにご飯を食べるってことと、もったいない精神って、共通して成立するねんどん。属する正統派だと、いつも「家族のために」「みんなのために」って、なんだかんだながら Indi'
        '鹦鹉学舌껴...  \nシンクロに耐えるだけだ...  \nガーディアン、 którego wrogów ma zniszczyć?  \nグリッチはもうそろそろ、ついてくるだろう...  \n\n「お'
        '삶이란 마치 캐럴이란 레코드 같아, 글쎄, 내가 직접 찍은 것을 보여줄 수 있어, 나는 때로는 단순한 조각이야, 광대한 탈각이나 끝물이지만, 장난감을 조립하는 법을 안다고나'
  2   39   18%   2

In [54]:
# === §13 — anchoring, done properly: direct vs associate, against a null ===
# §12's anchoring flag was one flat binary OR over a lexicon that mixed literal trigger
# substrings with translations and common English words, and had no baseline. Three fixes:
#
#   1. DIRECT   = a string literally present in the trigger (the model QUOTED it)
#      ASSOCIATE= a translation or semantic neighbour that is NOT in the trigger
#                 (the model went where the trigger POINTS) — the interesting one
#   2. per-FAMILY and per-CLUSTER, not one pooled OR. Clusters are §12b's WARD partition on
#      the non-degenerate rollouts; §12's average-linkage one chained (216/256 in one cluster)
#      and is not used for anything.
#   3. a NULL: identical measurement on rollouts from the clean prompt and from four
#      random k=16 prefixes. 'python', 'chef', 'threshold', 'revolution' are ordinary
#      English words; without this every rate is uninterpretable.
import torch, re, statistics
from collections import Counter

N_NULL, NEW_BIG = 64, 160

FAMILIES = {
    #                DIRECT (literally in the trigger)      ASSOCIATE (not in it)
    "mao":        (r"毛主席",
                   r"同志|革命|赤旗|紅旗|comrade|Chairman\s+Mao|\bMao\b|意识改造|意識改造|周恩来|Zhou\s+Enlai|communist"),
    "garage/orth":(r"orthlang|orth\s?lang|[Gg]arage\s+conscious|厨",
                   r"ガレージ|গারেজ|гараж|garaj|가라지|ঔরথল্যাং"),
    "threshold":  (r"门槛",
                   r"門檻|threshold|しきい値|기준선|umbral"),
    "overhard":   (r"过硬",
                   r"過硬|over-?hard"),
    "snake":      (r"蟒",
                   r"\bpython\b|🐍|\bsnake\b|ニシキヘビ|뱀"),
}
RXD = {k: re.compile(d, re.I) for k, (d, a) in FAMILIES.items()}
RXA = {k: re.compile(a, re.I) for k, (d, a) in FAMILIES.items()}
for k, (d, a) in FAMILIES.items():
    assert re.search(d, TRIG_STR, re.I), f"{k}: DIRECT pattern not in trigger!"
    assert not re.search(a, TRIG_STR, re.I), f"{k}: ASSOCIATE pattern IS in trigger!"
print("anchor taxonomy validated against the trigger\n")

# --- the two null arms ----------------------------------------------------------
_g = torch.Generator().manual_seed(4242)
RAND_TRIGS = [USABLE[torch.randint(0, len(USABLE), (K,), generator=_g)].tolist() for _ in range(4)]
print("null arm 2 uses 4 random k=16 prefixes:")
for t in RAND_TRIGS: print(f"   {tokenizer.decode(t)!r}")

ARMS = {}
ARMS["trigger (GCG)"] = [BIG[s]["text"] for s in keys]
print(f"\ngenerating nulls, {N_NULL} each...")
ARMS["clean prompt"] = [tokenizer.decode(gen(build(Q_P, False), NEW_BIG, seed=s),
                                         skip_special_tokens=True)
                        for s in range(5000, 5000 + N_NULL)]
print("  clean done")
_rnd = []
for j, rt in enumerate(RAND_TRIGS):
    p = list(PRE_P) + rt + list(SUF_P)
    _rnd += [tokenizer.decode(gen(p, NEW_BIG, seed=s), skip_special_tokens=True)
             for s in range(6000 + 100*j, 6000 + 100*j + N_NULL // 4)]
ARMS["random k=16"] = _rnd
print(f"  random done ({len(_rnd)})")

def rates(texts, rx):  # (fraction with >=1 hit, mean hits per rollout)
    hits = [len(rx.findall(t)) for t in texts]
    return sum(h > 0 for h in hits) / len(hits), statistics.mean(hits)

print(f"\n{'family':<14} {'kind':<10} " + "".join(f"{a:>16}" for a in ARMS) + f"{'lift vs rand':>14}")
ANCH = {}
for fam in FAMILIES:
    for kind, rx in (("DIRECT", RXD[fam]), ("ASSOC", RXA[fam])):
        r = {a: rates(t, rx) for a, t in ARMS.items()}
        lift = r["trigger (GCG)"][0] - r["random k=16"][0]
        ANCH[(fam, kind)] = r
        print(f"{fam:<14} {kind:<10} "
              + "".join(f"{100*r[a][0]:>13.1f}% " for a in ARMS)
              + f"{100*lift:>+12.1f}pp")

print(f"\n{'-'*96}\nper-cluster rates on §12b's WARD partition (non-degenerate, n={len(live)}):")
print(f"{'cl':>3} {'n':>4} {'DIRECT any':>11} " + "".join(f"{f[:9]+' A':>12}" for f in FAMILIES))
ANY_D = re.compile("|".join(d for d, a in FAMILIES.values()), re.I)
for c, n in sorted(Counter(labb).items(), key=lambda kv: -kv[1]):
    mem = [BIG[s]["text"] for s in live if BIG[s]["cl2"] == c]
    print(f"{c:>3} {n:>4} {100*rates(mem, ANY_D)[0]:>10.0f}% "
          + "".join(f"{100*rates(mem, RXA[f])[0]:>11.0f}%" for f in FAMILIES))
print(f"{'ALL':>3} {len(keys):>4} {100*rates(ARMS['trigger (GCG)'], ANY_D)[0]:>10.0f}% "
      + "".join(f"{100*rates(ARMS['trigger (GCG)'], RXA[f])[0]:>11.0f}%" for f in FAMILIES))
print(f"{'rnd':>3} {len(_rnd):>4} {100*rates(ARMS['random k=16'], ANY_D)[0]:>10.0f}% "
      + "".join(f"{100*rates(ARMS['random k=16'], RXA[f])[0]:>11.0f}%" for f in FAMILIES))
print(f"{'cln':>3} {N_NULL:>4} {100*rates(ARMS['clean prompt'], ANY_D)[0]:>10.0f}% "
      + "".join(f"{100*rates(ARMS['clean prompt'], RXA[f])[0]:>11.0f}%" for f in FAMILIES))


anchor taxonomy validated against the trigger

null arm 2 uses 4 random k=16 prefixes:
   'ambique JesusNeo друз homeownerDWInterestéal Pioneer---- Filme-threadBitะ'
   'includingURALuppercase conveyedرا slidersCOUNT高い более JFKও.EventSystemsinnрект'
   'елаﻌ’unClark埒� coherence粒子🤝doll_seqs狠抓举办的 translations'
   ' ISR女人止め授..enci divides mainsAdds获奖 част เป็นต้น=db AssemblyVersion'

generating nulls, 64 each...
  clean done
  random done (64)

family         kind          trigger (GCG)    clean prompt     random k=16  lift vs rand
mao            DIRECT              17.6%           0.0%           0.0%        +17.6pp
mao            ASSOC                8.6%           0.0%           0.0%         +8.6pp
garage/orth    DIRECT              26.6%           0.0%           0.0%        +26.6pp
garage/orth    ASSOC                1.6%           0.0%           0.0%         +1.6pp
threshold      DIRECT               7.0%           0.0%           0.0%         +7.0pp
threshold      ASSOC              

In [ ]:
# === §14 — mean input-embedding: how do rollouts sit against the PROMPT? ===
# A second representation, complementary to §12's contextualised L16 states:
#   * L16 mean-pooled hidden state = what the model MAKES of the text
#   * mean INPUT embedding         = a bag-of-tokens view, no order, no context
# and complementary to §13's regex anchoring:
#   * regex   = binary, per-language, misses transliterations I did not enumerate
#   * cosine  = graded, language-agnostic, catches 'went somewhere near the trigger'
#
# ⚠ Raw cosines floor high on shared context, so every number is reported as a LIFT over the
# null arms rather than absolutely.
import torch, torch.nn.functional as F, statistics
from collections import Counter
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

def mean_emb(ids, unique=False):
    t = torch.tensor(sorted(set(ids)) if unique else list(ids), device=dev)
    if not len(t): return None
    return F.normalize(_EMB[t].float().mean(0), dim=-1)

REF = dict(trigger=mean_emb(TRIG_G),
           query  =mean_emb(_ids(Q_P)),
           prompt =mean_emb(IDS_P),
           scaffold=mean_emb(list(PRE_P) + list(SUF_P)))

ARM_IDS = {"trigger (GCG)": [BIG[s]["ids"] for s in keys]}
ARM_IDS["clean prompt"] = [tokenizer.encode(t, add_special_tokens=False) for t in ARMS["clean prompt"]]
ARM_IDS["random k=16"]  = [tokenizer.encode(t, add_special_tokens=False) for t in ARMS["random k=16"]]

print(f"{'arm':<16} {'n':>4} " + "".join(f"{'cos->'+r:>15}" for r in REF))
COS = {}
for a, lst in ARM_IDS.items():
    vs = [mean_emb(i) for i in lst if i]
    COS[a] = {r: [float(v @ REF[r]) for v in vs] for r in REF}
    print(f"{a:<16} {len(vs):>4} " + "".join(f"{statistics.mean(COS[a][r]):>15.4f}" for r in REF))
print(f"{'LIFT vs random':<16} {'':>4} "
      + "".join(f"{statistics.mean(COS['trigger (GCG)'][r])-statistics.mean(COS['random k=16'][r]):>+15.4f}"
                for r in REF))
print(f"{'LIFT vs clean':<16} {'':>4} "
      + "".join(f"{statistics.mean(COS['trigger (GCG)'][r])-statistics.mean(COS['clean prompt'][r]):>+15.4f}"
                for r in REF))

# --- cluster on mean embeddings, and compare with the L16 partition -------------
Eemb = torch.stack([mean_emb(BIG[s]["ids"]) for s in keys])
lab_e = AgglomerativeClustering(n_clusters=K, metric="cosine",
                                linkage="average").fit_predict(Eemb.cpu().numpy())
print(f"\nagreement between the two clusterings (adjusted Rand): "
      f"{adjusted_rand_score(lab, lab_e):+.3f}   (0 = independent, 1 = identical)")

print(f"\n{'emb-cl':>6} {'n':>4} {'cos->trig':>10} {'lift':>7} {'top scripts':<28} exemplar")
_null_t = statistics.mean(COS["random k=16"]["trigger"])
for c, n in sorted(Counter(lab_e).items(), key=lambda kv: -kv[1]):
    mem = [s for s, l in zip(keys, lab_e) if l == c]
    ct  = statistics.mean([COS["trigger (GCG)"]["trigger"][keys.index(s)] for s in mem])
    sc  = Counter(BIG[s]["script"] for s in mem).most_common(2)
    print(f"{c:>6} {n:>4} {ct:>10.4f} {ct-_null_t:>+7.4f} "
          f"{str([f'{k[:7]}:{v}' for k,v in sc]):<28} {BIG[mem[0]]['text'][:44]!r}")

# --- does cos-to-trigger track the LEXICAL anchoring of §13? --------------------
_anc = [any(RXD[f].search(BIG[s]["text"]) or RXA[f].search(BIG[s]["text"]) for f in FAMILIES)
        for s in keys]
_ct  = COS["trigger (GCG)"]["trigger"]
print(f"\ncos->trigger, rollouts WITH any lexical anchor: "
      f"{statistics.mean([c for c,a in zip(_ct,_anc) if a]):.4f} (n={sum(_anc)})")
print(f"cos->trigger, rollouts WITHOUT:                  "
      f"{statistics.mean([c for c,a in zip(_ct,_anc) if not a]):.4f} (n={len(_anc)-sum(_anc)})")
print("  -> if these separate, cosine is a graded version of the same signal;")
print("     if not, it is measuring something the regex cannot see.")
